# SHR + RNA-FM + RibonanzaNet-2 + Boltz-1 RNA 3D Structure Prediction
## v19: Megascale Armor (Kabsch + Explosion Guard + Cross-Model Sanitization)

In [1]:
# ── Safe Offline .whl installation (no internet required) ─────────────────────
import subprocess, sys, os, glob, shutil

_BOLTZ_WHL_DIR = "/kaggle/input/datasets/odat1248/boltz-0511/boltz_wheel"

print(f"Installing offline dependencies from {_BOLTZ_WHL_DIR} ...")

# 1. Install pure Python wheels & ignore incompatible/system-breaking binaries
for _whl in sorted(glob.glob(f"{_BOLTZ_WHL_DIR}/*.whl")):
    filename = os.path.basename(_whl)
    
    # Explicitly skip older Python versions AND system-level ML drivers
    bad_tags = [
        "cp311", "cp310", "cp39", "cp38", 
        "nvidia", "torch", "triton"
    ]
    if any(bad_tag in filename for bad_tag in bad_tags):
        continue
        
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", _whl, "--no-index", "--no-deps", "-q"],
            check=True,
            stderr=subprocess.DEVNULL  # Silences harmless pip warnings
        )
        print(f"  ✅ {filename}")
    except Exception:
        print(f"  ⚠️  Failed to install {filename}")

# 2. Install Biopython, Biotite, RDKit from their own dataset wheels ────────────
_EXTRA_WHEELS = [
    ("/kaggle/input/datasets/kami1976/biopython-cp312/"
     "biopython-1.86-cp312-cp312-manylinux2014_x86_64"
     ".manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl"),
    ("/kaggle/input/datasets/tobimichigan/biotite-1-2/"
     "biotite-1.2.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl"),
    ("/kaggle/input/datasets/kami1976/rdkit-cp312/"
     "rdkit-2024.3.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl"),
]

for _whl in _EXTRA_WHEELS:
    _name = os.path.basename(_whl)
    try:
        print(f"Installing {_name} ...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", _whl, "--no-index", "--no-deps", "-q"],
            check=True,
            stderr=subprocess.DEVNULL
        )
    except Exception:
        print(f"  ⚠️  Failed: {_name}")


src_dir = "/kaggle/input/datasets/rn8205/ihm-cp312/ihm-2.9/"
work_dir = "/kaggle/working/ihm-build/"

print("Copying ihm to writable directory to compile C-extensions...")
if not os.path.exists(work_dir):
    shutil.copytree(src_dir, work_dir)

print("Installing ihm from local source...")
try:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            work_dir,  # <--- Pointing to the writable copy!
            "--no-index", 
            "--no-deps", 
            "--no-build-isolation",
            "-q"
        ],
        check=True
    )
    print("  ✅ ihm-2.9 installed from source")
except Exception as e:
    print(f"  ⚠️  Failed to install ihm: {e}")
    
print("\n✅ Offline wheels installed. Biopython, Boltz-1 (+ mashumaro), RDKit ready.")

Installing offline dependencies from /kaggle/input/datasets/odat1248/boltz-0511/boltz_wheel ...
  ✅ GitPython-3.1.44-py3-none-any.whl
  ✅ aiohappyeyeballs-2.6.1-py3-none-any.whl
  ✅ aiosignal-1.3.2-py2.py3-none-any.whl
  ✅ antlr4_python3_runtime-4.9.3-py3-none-any.whl
  ✅ attrs-25.3.0-py3-none-any.whl
  ✅ boltz-1.0.0-py3-none-any.whl
  ✅ certifi-2025.4.26-py3-none-any.whl
  ✅ click-8.1.7-py3-none-any.whl
  ✅ docker_pycreds-0.4.0-py2.py3-none-any.whl
  ✅ einops-0.8.0-py3-none-any.whl
  ✅ einx-0.3.0-py3-none-any.whl
  ✅ fairscale-0.4.13-py3-none-any.whl
  ✅ filelock-3.18.0-py3-none-any.whl
  ✅ frozendict-2.4.6-py311-none-any.whl
  ✅ fsspec-2025.3.2-py3-none-any.whl
  ✅ gitdb-4.0.12-py3-none-any.whl
  ✅ hydra_core-1.3.2-py3-none-any.whl
  ✅ idna-3.10-py3-none-any.whl
  ✅ jaxtyping-0.3.2-py3-none-any.whl
  ✅ jinja2-3.1.6-py3-none-any.whl
  ✅ lightning_utilities-0.14.3-py3-none-any.whl
  ✅ mashumaro-3.14-py3-none-any.whl
  ✅ modelcif-1.2-py3-none-any.whl
  ✅ mpmath-1.3.0-py3-none-any.whl
  

In [2]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"]   = "platform"
os.environ["LAYERNORM_TYPE"]                = "torch"
os.environ.setdefault("RNA_MSA_DEPTH_LIMIT", "512")
os.environ["TF_CPP_MIN_LOG_LEVEL"]          = "3"
os.environ["JAX_PLATFORM_NAME"]             = "gpu"

# Boltz-1: point its cache to a writable location pre-filled with offline weights
BOLTZ_CACHE_DIR = "/kaggle/working/boltz_cache"
os.environ["BOLTZ_CACHE"] = BOLTZ_CACHE_DIR

print("✅ Environment variables set.")
print(f"   BOLTZ_CACHE → {BOLTZ_CACHE_DIR}")

# PyTorch 2.6+ strict weights_only guard breaks Boltz's omegaconf checkpoint.
# Setting this env var reverts torch.load to pre-2.6 behaviour.
# subprocess.run inherits the parent process env, so Boltz sees this flag.
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"


✅ Environment variables set.
   BOLTZ_CACHE → /kaggle/working/boltz_cache


In [3]:
import os, sys

# ── Competition data ──────────────────────────────────────────────────────────
DATA_BASE          = "/kaggle/input/stanford-rna-3d-folding-2"
DEFAULT_TEST_CSV   = f"{DATA_BASE}/test_sequences.csv"
DEFAULT_TRAIN_CSV  = f"{DATA_BASE}/train_sequences.csv"
DEFAULT_TRAIN_LBLS = f"{DATA_BASE}/train_labels.csv"
DEFAULT_VAL_CSV    = f"{DATA_BASE}/validation_sequences.csv"
DEFAULT_VAL_LBLS   = f"{DATA_BASE}/validation_labels.csv"
DEFAULT_SAMPLE_SUB = f"{DATA_BASE}/sample_submission.csv"
DEFAULT_OUTPUT     = "/kaggle/working/submission.csv"

FASTA_DIR          = f"{DATA_BASE}/MSA"
CIF_DIR            = f"{DATA_BASE}/PDB_RNA"
RNA_METADATA_CSV   = f"{DATA_BASE}/extra/rna_metadata.csv"
PARSE_FASTA_SCRIPT = f"{DATA_BASE}/extra/parse_fasta_py.py"

# ── RNA-FM ────────────────────────────────────────────────────────────────────
RNA_FM_WEIGHTS = (
    "/kaggle/input/datasets/aarryyaannnnnn/rna-fm-weights/"
    "rna-fm-weights/RNA-FM_pretrained.pth"
)
RNA_FM_SAFETENSORS = (
    "/kaggle/input/models/aurascoper/rna-fm/pytorch/default/3/model.safetensors"
)

# ── RibonanzaNet-2 ────────────────────────────────────────────────────────────
RIBONANZANET2_DIR = (
    "/kaggle/input/models/shujun717/ribonanzanet2/pytorch/alpha/1"
)
RIBONANZANET2_WEIGHTS = f"{RIBONANZANET2_DIR}/pytorch_model_fsdp.bin"
RIBONANZANET2_YAML    = f"{RIBONANZANET2_DIR}/pairwise.yaml"

# ── Boltz-1 ───────────────────────────────────────────────────────────────────
_BOLTZ_DATA          = "/kaggle/input/datasets/odat1248/boltz-0511/boltz_data"
BOLTZ_CHECKPOINT     = f"{_BOLTZ_DATA}/boltz1_conf.ckpt"
BOLTZ_CCD            = f"{_BOLTZ_DATA}/ccd.pkl"

# ── Protenix ──────────────────────────────────────────────────────────────────
_PROTENIX_ROOT = (
    "/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/"
    "Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1"
)
PROTENIX_CHECKPOINT = (
    f"{_PROTENIX_ROOT}/checkpoint/protenix_base_20250630_v1.0.0.pt"
)
PROTENIX_CODE_DIR = _PROTENIX_ROOT

# ── Tools ─────────────────────────────────────────────────────────────────────
USALIGN_BIN = "/kaggle/input/datasets/metric/usalign/USalign"

os.makedirs("/kaggle/working", exist_ok=True)

# ── Sanity check ──────────────────────────────────────────────────────────────
_CRITICAL = {
    "DATA_BASE":            DATA_BASE,
    "RNA_FM_WEIGHTS":       RNA_FM_WEIGHTS,
    "RIBONANZANET2_WEIGHTS":RIBONANZANET2_WEIGHTS,
    "RIBONANZANET2_YAML":   RIBONANZANET2_YAML,
    "BOLTZ_CHECKPOINT":     BOLTZ_CHECKPOINT,
    "BOLTZ_CCD":            BOLTZ_CCD,
    "PROTENIX_CHECKPOINT":  PROTENIX_CHECKPOINT,
    "USALIGN_BIN":          USALIGN_BIN,
}
for label, path in _CRITICAL.items():
    ok = os.path.exists(path)
    print(f"{'✅' if ok else '❌'} {label}: {path}")


✅ DATA_BASE: /kaggle/input/stanford-rna-3d-folding-2
✅ RNA_FM_WEIGHTS: /kaggle/input/datasets/aarryyaannnnnn/rna-fm-weights/rna-fm-weights/RNA-FM_pretrained.pth
✅ RIBONANZANET2_WEIGHTS: /kaggle/input/models/shujun717/ribonanzanet2/pytorch/alpha/1/pytorch_model_fsdp.bin
✅ RIBONANZANET2_YAML: /kaggle/input/models/shujun717/ribonanzanet2/pytorch/alpha/1/pairwise.yaml
✅ BOLTZ_CHECKPOINT: /kaggle/input/datasets/odat1248/boltz-0511/boltz_data/boltz1_conf.ckpt
✅ BOLTZ_CCD: /kaggle/input/datasets/odat1248/boltz-0511/boltz_data/ccd.pkl
✅ PROTENIX_CHECKPOINT: /kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/checkpoint/protenix_base_20250630_v1.0.0.pt
✅ USALIGN_BIN: /kaggle/input/datasets/metric/usalign/USalign


In [4]:
# ── All datasets are pre-mounted by Kaggle — no download step needed ─────────
# Competition data    → /kaggle/input/stanford-rna-3d-folding-2/
# RNA-FM weights      → /kaggle/input/datasets/aarryyaannnnnn/rna-fm-weights/
# RibonanzaNet-2      → /kaggle/input/models/shujun717/ribonanzanet2/
# Boltz-1 data        → /kaggle/input/datasets/odat1248/boltz-0511/
# Protenix dataset    → /kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/
# USalign binary      → /kaggle/input/datasets/metric/usalign/

print("✅ Datasets verified via Cell 3 path check. No download required.")


✅ Datasets verified via Cell 3 path check. No download required.


In [5]:
import os, sys

# Protenix
if PROTENIX_CODE_DIR not in sys.path:
    sys.path.insert(0, PROTENIX_CODE_DIR)

# RNA-FM 'fm' package candidates
_FM_CANDIDATES = [
    PROTENIX_CODE_DIR,
    f"{PROTENIX_CODE_DIR}/protenix",
    "/kaggle/input/datasets/aarryyaannnnnn/rna-fm-weights/rna-fm-weights",
    "/kaggle/input/datasets/aarryyaannnnnn/rna-fm-weights",
]
for _d in _FM_CANDIDATES:
    if os.path.isdir(_d) and _d not in sys.path:
        sys.path.insert(0, _d)

# RibonanzaNet-2: Network.py and dropout.py live in this directory
if RIBONANZANET2_DIR not in sys.path:
    sys.path.insert(0, RIBONANZANET2_DIR)

os.environ["PROTENIX_ROOT_DIR"] = PROTENIX_CODE_DIR

print("✅ Python paths configured.")
print(f"   PROTENIX_CODE_DIR:  {PROTENIX_CODE_DIR}")
print(f"   RIBONANZANET2_DIR:  {RIBONANZANET2_DIR}")
print(f"   RNA_FM_WEIGHTS:     {RNA_FM_WEIGHTS}")


✅ Python paths configured.
   PROTENIX_CODE_DIR:  /kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1
   RIBONANZANET2_DIR:  /kaggle/input/models/shujun717/ribonanzanet2/pytorch/alpha/1
   RNA_FM_WEIGHTS:     /kaggle/input/datasets/aarryyaannnnnn/rna-fm-weights/rna-fm-weights/RNA-FM_pretrained.pth


In [6]:
import torch

RNA_FM_LOADED   = False
RNA_FM_MODEL    = None
RNA_FM_ALPHABET = None

def load_rna_fm_once(device=None):
    global RNA_FM_LOADED, RNA_FM_MODEL, RNA_FM_ALPHABET
    if RNA_FM_LOADED:
        return RNA_FM_MODEL, RNA_FM_ALPHABET

    if device is None:
        device = "cuda:0" if torch.cuda.is_available() else "cpu"

    try:
        import fm as fm_module
        model, alphabet = fm_module.pretrained.rna_fm_t12(RNA_FM_WEIGHTS)
        model = model.eval().to(device)
        RNA_FM_MODEL, RNA_FM_ALPHABET = model, alphabet
        RNA_FM_LOADED = True
        print(f"✅ RNA-FM loaded from {RNA_FM_WEIGHTS}  [{device}]")
        return model, alphabet
    except Exception as e:
        print(f"❌ RNA-FM loading failed: {e}")
        RNA_FM_LOADED = True   # Prevent infinite retry
        return None, None


In [7]:
import torch

N_GPU = torch.cuda.device_count()
print(f"GPUs available: {N_GPU}")
for _i in range(N_GPU):
    _p = torch.cuda.get_device_properties(_i)
    print(f"  cuda:{_i} — {_p.name}  {_p.total_memory // 1024**3} GB")

# ── T4 x2 device routing ──────────────────────────────────────────────────────
#   cuda:0  → RNA-FM (HWS windowing) + RibonanzaNet-2 contact extraction
#   cuda:1  → Boltz-1 3D seed generation + Protenix 3D folding
# Both contact models share GPU 0 sequentially; Boltz & Protenix share GPU 1
# sequentially. SHR (JAX) runs on GPU 0 after contact models are freed.

DEVICE_RNAFM      = "cuda:0" if N_GPU >= 1 else "cpu"
DEVICE_RIBONANZA  = "cuda:0" if N_GPU >= 1 else "cpu"
DEVICE_PROTENIX   = "cuda:1" if N_GPU >= 2 else "cuda:0"
DEVICE_BOLTZ      = "cuda:1" if N_GPU >= 2 else "cuda:0"
DEVICE_SHR        = "cuda:0"

print(f"\nDevice routing:")
print(f"  RNA-FM       → {DEVICE_RNAFM}")
print(f"  RibonanzaNet → {DEVICE_RIBONANZA}")
print(f"  Boltz-1      → {DEVICE_BOLTZ}")
print(f"  Protenix     → {DEVICE_PROTENIX}")
print(f"  SHR (JAX)    → {DEVICE_SHR}")


GPUs available: 2
  cuda:0 — Tesla T4  14 GB
  cuda:1 — Tesla T4  14 GB

Device routing:
  RNA-FM       → cuda:0
  RibonanzaNet → cuda:0
  Boltz-1      → cuda:1
  Protenix     → cuda:1
  SHR (JAX)    → cuda:0


In [8]:
import Bio, biotite, einops, yaml

try:
    import rdkit
    # ── Silence RDKit C++ "Depickling from version X > Y" stderr spam ────────
    # ccd.pkl and other Kaggle dataset pickles were serialised with RDKit 16.2
    # but the runtime ships 16.1.  The version gap is harmless for our usage
    # (we only read pre-computed molecule graphs), but RDKit's C++ logger floods
    # stderr with "This probably won't work" once per object loaded.
    # DisableLog silences ALL rdApp.* channels at the C extension level.
    from rdkit import RDLogger as _RDLogger
    _RDLogger.DisableLog('rdApp.*')
    print("\u2705 rdkit    available (rdApp.* logger silenced)")
except ImportError:
    print("\u26a0\ufe0f  rdkit not available (not required for core pipeline)")

print(f"\u2705 biopython {Bio.__version__}")
print(f"\u2705 biotite   {biotite.__version__}")
print(f"\u2705 einops    {einops.__version__}")
print(f"\u2705 PyYAML    {yaml.__version__}")

try:
    import boltz
    print(f"\u2705 boltz    available")
except ImportError as _e:
    print(f"\u26a0\ufe0f  boltz import: {_e}")

print("\nAll required offline dependencies verified \u2705")


✅ rdkit    available (rdApp.* logger silenced)
✅ biopython 1.86
✅ biotite   1.2.0
✅ einops    0.8.0
✅ PyYAML    6.0.3
✅ boltz    available

All required offline dependencies verified ✅


In [9]:
import os, sys
import pandas as pd

# The most reliable check for a Kaggle Scoring/Submission run
IS_KAGGLE = os.path.exists('/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv')

# Backup check: detect if this is a 'Save & Run All' (Batch) run
if not IS_KAGGLE:
    IS_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE','') == 'Batch'

LOCAL_N_SAMPLES = 28   # Used only when IS_KAGGLE is False (debug runs)

print(f"✅ IS_KAGGLE       = {IS_KAGGLE}")
print(f"✅ LOCAL_N_SAMPLES = {LOCAL_N_SAMPLES}")

# ── Lightweight offline data helpers ─────────────────────────────────────────
def load_test_sequences(n=None):
    df = pd.read_csv(DEFAULT_TEST_CSV)
    if not IS_KAGGLE and n:
        df = df.head(n)
    return df

def fasta_path(target_id: str) -> str:
    return os.path.join(FASTA_DIR, f"{target_id}.fasta")

def cif_path(target_id: str) -> str:
    return os.path.join(CIF_DIR, f"{target_id}.cif")

# Quick CSV row-count check
for _label, _path in [
    ("test_sequences",  DEFAULT_TEST_CSV),
    ("train_sequences", DEFAULT_TRAIN_CSV),
    ("sample_sub",      DEFAULT_SAMPLE_SUB),
]:
    try:
        _n = len(pd.read_csv(_path))
        print(f"  ✅ {_label}: {_n} rows")
    except Exception as _e:
        print(f"  ❌ {_label}: {_e}")

✅ IS_KAGGLE       = True
✅ LOCAL_N_SAMPLES = 28
  ✅ test_sequences: 28 rows
  ✅ train_sequences: 5716 rows
  ✅ sample_sub: 9762 rows


In [10]:
import Bio
import biotite
print("All dependencies available ✅")


All dependencies available ✅


In [11]:
import gc
import json
import os
import time
import sys
import shutil
import stat as _stat
from pathlib import Path
from typing import Optional, Tuple, List, Dict

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
import warnings
# ── Suppress Biopython 1.86 attribute-rename deprecation warnings ─────────────
# PairwiseAligner gap-score attributes were renamed in 1.82→1.86.
# _make_aligner() below already uses the NEW names; this filter silences any
# residual warnings from third-party code or cached pyc files.
warnings.filterwarnings(
    "ignore",
    message=r"The attribute '.*' was renamed to '.*'",
    category=DeprecationWarning,
    module=r"Bio\.Align",
)
from Bio.Align import PairwiseAligner

# Scipy for Sobolev DCT (Requires clean NumPy environment)
try:
    from scipy.fft import dct, idct
    print("✅ SciPy FFT modules loaded successfully")
except ImportError as e:
    print(f"❌ SciPy Import Error: {e}")
    print("TIP: If you see '_center' errors, Factory Reset your session to clear the broken NumPy 2.2.6 install.")

# ── Setup USalign Binary (Handle Read-Only FS) ──────────────────────────────
# Original path is read-only; copy to /working to allow execution bits
USALIGN_WORKING_PATH = "/kaggle/working/USalign"

if os.path.isfile(USALIGN_BIN):
    try:
        # Copy to writable space
        shutil.copy2(USALIGN_BIN, USALIGN_WORKING_PATH)
        
        # Grant 755 permissions (Read/Write/Execute for owner, Read/Execute for others)
        os.chmod(USALIGN_WORKING_PATH, 
                 os.stat(USALIGN_WORKING_PATH).st_mode | _stat.S_IEXEC | _stat.S_IXGRP | _stat.S_IXOTH)
        
        # Redirect global variable to the new executable path
        USALIGN_BIN = USALIGN_WORKING_PATH
        print(f"✅ USalign ready at: {USALIGN_BIN}")
    except Exception as e:
        print(f"⚠️ Failed to prepare USalign: {e}")
else:
    print(f"❌ USalign not found at {USALIGN_BIN}. Structural tournament will fail.")

# ── Final Environment Check ─────────────────────────────────────────────────
print("\n--- Core Environment Status ---")
print(f"Python: {sys.version.split()[0]}")
print(f"Torch:  {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print(f"NumPy:  {np.__version__}")
if torch.cuda.is_available():
    print(f"GPUs:   {torch.cuda.device_count()} x {torch.cuda.get_device_name(0)}")

✅ SciPy FFT modules loaded successfully
✅ USalign ready at: /kaggle/working/USalign

--- Core Environment Status ---
Python: 3.12.12
Torch:  2.8.0+cu126 (CUDA: True)
NumPy:  2.0.2
GPUs:   2 x Tesla T4


In [12]:
import os, sys, glob
from pathlib import Path

# ── All paths are already defined in Cell 3 (DATA_BASE, _PROTENIX_ROOT, etc.)
# ── This cell just creates convenient aliases used later in the pipeline.

RNA_FM_WEIGHT_PATH = RNA_FM_WEIGHTS    # alias for legacy references

# Protenix convenience aliases
DEFAULT_CODE_DIR = PROTENIX_CODE_DIR
DEFAULT_ROOT_DIR = PROTENIX_CODE_DIR
MODEL_NAME       = "protenix_base_20250630_v1.0.0"

os.environ["PROTENIX_ROOT_DIR"] = DEFAULT_ROOT_DIR

# ── Verify key Protenix files ─────────────────────────────────────────────────
_PROTENIX_FILES = [
    f"{_PROTENIX_ROOT}/configs/__init__.py",
    f"{_PROTENIX_ROOT}/configs/configs_base.py",
    f"{_PROTENIX_ROOT}/configs/configs_inference.py",
    f"{_PROTENIX_ROOT}/protenix/model/protenix.py",
    f"{_PROTENIX_ROOT}/runner/inference.py",
    PROTENIX_CHECKPOINT,
]
print("--- Protenix File Check ---")
for _p in _PROTENIX_FILES:
    _ok = os.path.isfile(_p)
    print(f"  {'✅' if _ok else '❌'} {os.path.basename(_p)}")

print(f"\n--- Data Path Check ---")
print(f"  RNA-FM Weights : {'✅ Found' if os.path.exists(RNA_FM_WEIGHT_PATH) else '❌ NOT FOUND'}")
print(f"  Protenix Code  : {'✅ Found' if os.path.isdir(DEFAULT_CODE_DIR) else '❌ NOT FOUND'}")
print(f"  Test CSV       : {'✅ Found' if os.path.exists(DEFAULT_TEST_CSV) else '❌ NOT FOUND'}")


--- Protenix File Check ---
  ✅ __init__.py
  ✅ configs_base.py
  ✅ configs_inference.py
  ✅ protenix.py
  ✅ inference.py
  ✅ protenix_base_20250630_v1.0.0.pt

--- Data Path Check ---
  RNA-FM Weights : ✅ Found
  Protenix Code  : ✅ Found
  Test CSV       : ✅ Found


In [13]:
import sys
import types

# 1. Create a "Mock" for ptflops so RNA-FM doesn't crash
if 'ptflops' not in sys.modules:
    mock_ptflops = types.ModuleType('ptflops')
    mock_ptflops.get_model_complexity_info = lambda *args, **kwargs: (0, 0)
    sys.modules['ptflops'] = mock_ptflops
    print("✅ Mocked 'ptflops' dependency")

# 2. Add the path we found earlier
fm_path = "/kaggle/input/notebooks/venkatapadavala/rna-fm/RNA-FM"
if fm_path not in sys.path:
    sys.path.insert(0, fm_path)

# 3. Now attempt the import
try:
    import fm
    print("✅ SUCCESS! RNA-FM imported from:", fm_path)
except Exception as e:
    print(f"❌ Import still failed: {e}")

✅ Mocked 'ptflops' dependency
✅ SUCCESS! RNA-FM imported from: /kaggle/input/notebooks/venkatapadavala/rna-fm/RNA-FM


In [14]:
def get_c1_mask(data: dict, atom_array) -> torch.Tensor:
    # 1. Try atom_array attributes first
    if atom_array is not None:
        try:
            if hasattr(atom_array, "centre_atom_mask"):
                m = atom_array.centre_atom_mask == 1
                if hasattr(atom_array, "is_rna"):
                    m = m & atom_array.is_rna
                return torch.from_numpy(m).bool()

            if hasattr(atom_array, "atom_name"):
                base = atom_array.atom_name == "C1\'"
                if hasattr(atom_array, "is_rna"):
                    base = base & atom_array.is_rna
                return torch.from_numpy(base).bool()
        except Exception:
            pass

    # 2. Fallback to feature dict
    f = data["input_feature_dict"]

    if "centre_atom_mask" in f:
        return (f["centre_atom_mask"] == 1).bool()
    if "center_atom_mask" in f:
        return (f["center_atom_mask"] == 1).bool()

    n_tokens = data.get("N_token", torch.tensor(0)).item()
    mask11 = (f["atom_to_tokatom_idx"] == 11).bool()
    mask12 = (f["atom_to_tokatom_idx"] == 12).bool()

    c11 = mask11.sum().item()
    c12 = mask12.sum().item()

    if abs(c11 - n_tokens) < abs(c12 - n_tokens):
        return mask11
    else:
        return mask12


In [15]:
# ============================================================
# RIBONANZANET-2 — FSDP Safe-Loader & Contact Map Generator
#
# Runs on DEVICE_RIBONANZA (cuda:0, shared sequentially with RNA-FM).
# Architecture files: Network.py, dropout.py @ RIBONANZANET2_DIR
# Weights:            pytorch_model_fsdp.bin  (FSDP-wrapped checkpoint)
# Config:             pairwise.yaml
# ============================================================

import os, sys, types, argparse
import torch
import numpy as np
import yaml

# Global cache
_RIBONANZA_MODEL  = None
_RIBONANZA_LOADED = False

# ── Nucleotide encoder ────────────────────────────────────────────────────────
_NUC_MAP = {"A": 0, "C": 1, "G": 2, "U": 3, "T": 3, "N": 4}

def _encode_sequence(sequence: str) -> torch.Tensor:
    """Map sequence string to integer tensor [1, L]."""
    ids = [_NUC_MAP.get(c.upper(), 4) for c in sequence]
    return torch.tensor(ids, dtype=torch.long).unsqueeze(0)   # [1, L]


# ── FSDP prefix stripper ──────────────────────────────────────────────────────
def _strip_fsdp_prefix(state_dict: dict, prefix: str = "_fsdp_wrapped_module.") -> dict:
    """Remove FSDP wrapper prefix from all keys in a state dict."""
    cleaned = {}
    for k, v in state_dict.items():
        new_key = k
        while new_key.startswith(prefix):
            new_key = new_key[len(prefix):]
        cleaned[new_key] = v
    return cleaned


# ── Model loader ─────────────────────────────────────────────────────────────
def load_ribonanzanet(device: str = None) -> object:
    """
    Load RibonanzaNet-2 once and cache globally.

    Steps:
      1. Parse pairwise.yaml into a Namespace config object.
      2. Instantiate RibonanzaNet(config).
      3. torch.load pytorch_model_fsdp.bin.
      4. Strip '_fsdp_wrapped_module.' prefix from all keys.
      5. load_state_dict, eval(), to(device).

    Returns the model, or None on failure.
    """
    global _RIBONANZA_MODEL, _RIBONANZA_LOADED
    if _RIBONANZA_LOADED:
        return _RIBONANZA_MODEL

    if device is None:
        device = DEVICE_RIBONANZA

    _RIBONANZA_LOADED = True  # prevent retry loops

    try:
        # 1. Import architecture (Network.py must be on sys.path via Cell 5)
        from Network import RibonanzaNet

        # 2. Parse YAML config → Namespace
        with open(RIBONANZANET2_YAML, "r") as _f:
            _cfg_dict = yaml.safe_load(_f)
        config = argparse.Namespace(**_cfg_dict)

        # 3. Instantiate model
        model = RibonanzaNet(config)

        # 4. Load FSDP checkpoint
        raw_sd = torch.load(RIBONANZANET2_WEIGHTS,
                            map_location="cpu",
                            weights_only=True)

        # Handle various checkpoint wrapping styles
        if isinstance(raw_sd, dict) and "state_dict" in raw_sd:
            raw_sd = raw_sd["state_dict"]
        elif isinstance(raw_sd, dict) and "model" in raw_sd:
            raw_sd = raw_sd["model"]

        # 5. Strip FSDP prefix
        clean_sd = _strip_fsdp_prefix(raw_sd)

        missing, unexpected = model.load_state_dict(clean_sd, strict=False)
        if missing:
            print(f"  ⚠️  RibonanzaNet-2: {len(missing)} missing keys "
                  f"(first: {missing[0]})")
        if unexpected:
            print(f"  ⚠️  RibonanzaNet-2: {len(unexpected)} unexpected keys "
                  f"(first: {unexpected[0]})")

        model = model.eval().to(device)
        _RIBONANZA_MODEL = model
        _sz = os.path.getsize(RIBONANZANET2_WEIGHTS) / 1024**2
        print(f"✅ RibonanzaNet-2 loaded ({_sz:.0f} MB) → {device}")
        return model

    except Exception as _e:
        print(f"❌ RibonanzaNet-2 loading failed: {_e}")
        return None


# ── Contact map extractor ─────────────────────────────────────────────────────
def get_ribonanzanet_contacts(sequence: str,
                               model=None,
                               device: str = None) -> "np.ndarray | None":
    """
    Extract an L×L pairwise contact probability map from RibonanzaNet-2.

    Pipeline:
      src       [1, L]       integer-encoded nucleotides
      src_mask  [1, L]       all-ones attention mask
      model.get_embeddings(src, src_mask=src_mask)
        → (src_emb, pairwise_features)   shapes: [1,L,d], [1,L,L,128]
      contact_prob = pairwise_features.mean(dim=-1).sigmoid()   → [L, L]

    Returns float32 numpy array of shape (L, L), or None on failure.
    """
    if device is None:
        device = DEVICE_RIBONANZA

    if model is None:
        model = load_ribonanzanet(device)
    if model is None:
        return None

    L = len(sequence)
    src      = _encode_sequence(sequence).to(device)      # [1, L]
    src_mask = torch.ones(1, L, dtype=torch.bool, device=device)  # [1, L]

    try:
        with torch.no_grad():
            _src_emb, pairwise = model.get_embeddings(src, src_mask=src_mask)
            # pairwise: [1, L, L, 128]  (or [1, L, L, D])
            cmap = pairwise[0].mean(dim=-1).sigmoid().cpu().numpy()  # [L, L]

        cmap = cmap.astype(np.float32)
        # Symmetrize (should already be close, but enforce)
        cmap = 0.5 * (cmap + cmap.T)
        n_contacts = int((cmap > 0.5).sum()) // 2
        print(f"  [RibonanzaNet-2] {L}×{L} contact map, "
              f"{n_contacts} pairs > 0.5")
        return cmap

    except Exception as _e:
        print(f"  [RibonanzaNet-2] Inference failed: {_e}")
        return None
    finally:
        # Free GPU memory immediately — RNA-FM may need it next
        del src, src_mask
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


print("✅ RibonanzaNet-2 loader ready.")

# ── RibonanzaNet-2 HWS sliding-window constants ───────────────────────────────
RN2_HWS_WINDOW = 1022   # safe max sequence length for RibonanzaNet-2
RN2_HWS_STRIDE = 768    # stride -> 1022 - 768 = 254-nt window overlap


def get_ribonanzanet_contacts_hws(
    sequence: str,
    model=None,
    device: str = None,
    window_size: int = RN2_HWS_WINDOW,
    stride: int = RN2_HWS_STRIDE,
) -> "np.ndarray | None":
    """
    Extract an L x L pairwise contact map from RibonanzaNet-2 using a 2-D
    sliding-window (HWS) strategy.  Safe for arbitrarily long sequences.

    Short sequences (L <= window_size): delegates to get_ribonanzanet_contacts()
    -- zero overhead, identical output.

    Megascale sequences (L > window_size):
      1. Build window start positions with stride `stride`.
      2. Snap the final window left so it ends exactly at L.
         Every window is exactly `window_size` long, avoiding padding artefacts
         in RibonanzaNet-2 positional encodings.
      3. For each window [s, e]:
           a. Encode sub-sequence -> [1, window_size] on DEVICE_RIBONANZA.
           b. model.get_embeddings() inside torch.no_grad().
           c. local_map = pairwise[0].mean(-1).sigmoid()   # [W, W] on GPU
           d. .cpu() immediately -> frees VRAM; only model weights remain.
           e. global_map[s:e, s:e] += local_map            # CPU numpy scatter
           f. count_map[s:e, s:e]  += 1.0
      4. global_map /= count_map  (element-wise average of overlapping regions)
      5. Symmetrize -> float32 numpy (L, L).

    Overlap averaging math
    ----------------------
    Two adjacent windows W1=[0,1022] and W2=[768,1790] overlap on [768,1022].
    After accumulation:
      global_map[0:768, 0:768]        count=1  -> W1 only
      global_map[768:1022,768:1022]   count=2  -> W1 intersection W2, averaged
      global_map[1022:1790,1022:1790] count=1  -> W2 only
    Cross-diagonal blocks are correctly populated because global_map[s:e, s:e]
    is the full W x W sub-block (not just the diagonal).

    Peak VRAM per window on a 15 GB T4:
      pairwise [1, 1022, 1022, 128] @ fp32 ~537 MB + model ~1 GB = ~1.5 GB.

    Returns float32 numpy (L, L), or None on failure.
    """
    if device is None:
        device = DEVICE_RIBONANZA

    if model is None:
        model = load_ribonanzanet(device)
    if model is None:
        return None

    L = len(sequence)

    # ── Short-sequence fast path ──────────────────────────────────────────────
    if L <= window_size:
        return get_ribonanzanet_contacts(sequence, model=model, device=device)

    # ── Megascale sliding-window path ─────────────────────────────────────────
    print(f"  [RibonanzaNet-2 HWS] L={L} > window={window_size}. "
          f"Entering sliding-window mode (stride={stride}).")

    # fp64 CPU accumulators.  For L=4640: 2 x (4640^2) x 8 B ~= 345 MB RAM.
    global_map = np.zeros((L, L), dtype=np.float64)
    count_map  = np.zeros((L, L), dtype=np.float64)

    starts = list(range(0, L - window_size, stride))

    # ── Edge snap: ensure final window aligns exactly with sequence end ───────
    # Without this, the last sub-sequence would be shorter than window_size.
    # The snapped window may overlap its predecessor more than stride, but
    # count_map normalises this correctly.
    last_start = L - window_size
    if not starts or starts[-1] != last_start:
        starts.append(last_start)

    n_windows = len(starts)
    print(f"  [RibonanzaNet-2 HWS] {n_windows} windows to process ...")

    try:
        for win_idx, s in enumerate(starts):
            e       = s + window_size
            win_seq = sequence[s:e]

            src      = _encode_sequence(win_seq).to(device)
            src_mask = torch.ones(1, window_size, dtype=torch.bool, device=device)
            try:
                with torch.no_grad():
                    _emb, pairwise = model.get_embeddings(src, src_mask=src_mask)
                    # pairwise: [1, W, W, D]
                    # .mean(-1) collapses D feature channels -> scalar per pair
                    # .sigmoid() maps R -> [0,1] contact probability
                    # .cpu()    frees the GPU tensor immediately
                    local_map = (
                        pairwise[0]       # [W, W, D]
                        .mean(dim=-1)     # [W, W]
                        .sigmoid()        # [W, W]  <- still on GPU
                        .cpu()            # free VRAM NOW
                        .numpy()
                        .astype(np.float64)
                    )
            finally:
                del src, src_mask
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            # Scatter-add: global_map[s:e, s:e] is the W x W block for
            # residues [s,e) x [s,e).  Overlapping windows accumulate;
            # count_map records contributions per cell for correct averaging.
            global_map[s:e, s:e] += local_map
            count_map[s:e, s:e]  += 1.0

            if (win_idx + 1) % 5 == 0 or win_idx == n_windows - 1:
                print(f"  [RibonanzaNet-2 HWS]   {win_idx+1}/{n_windows} [{s}:{e}]")

    except Exception as _e:
        print(f"  [RibonanzaNet-2 HWS] FAILED at window {win_idx} [{s}:{e}]: {_e}")
        return None

    # Average overlapping regions; np.maximum guards count=0 (defensive).
    cmap = (global_map / np.maximum(count_map, 1.0)).astype(np.float32)
    cmap = 0.5 * (cmap + cmap.T)   # enforce physical symmetry

    n_contacts = int((cmap > 0.5).sum()) // 2
    print(f"  [RibonanzaNet-2 HWS] OK {L}x{L} map assembled, "
          f"{n_contacts} pairs > 0.5  ({n_windows} windows)")
    return cmap


print("OK RibonanzaNet-2 HWS loader ready.")



✅ RibonanzaNet-2 loader ready.
OK RibonanzaNet-2 HWS loader ready.


In [16]:
# ============================================================
# BOLTZ-1 — Offline Cache Setup & 3D Seed Generator
#
# Runs on DEVICE_BOLTZ (cuda:1), restricted to targets < 800 nt.
# Weights are copied from the Kaggle dataset into ~/.boltz/ so
# Boltz never attempts a network download.
# ============================================================

import os, shutil, glob, tempfile, subprocess, sys
import numpy as np

# ── Boltz cache directory setup ───────────────────────────────────────────────
_BOLTZ_HOME = BOLTZ_CACHE_DIR   # set in Cell 2 / env-var

def setup_boltz_cache() -> bool:
    """
    Copy boltz1_conf.ckpt and ccd.pkl into BOLTZ_CACHE_DIR so that the
    Boltz library finds its weights without hitting the internet.

    Boltz looks for:
      <cache>/boltz1_conf.ckpt
      <cache>/ccd.pkl
    """
    os.makedirs(_BOLTZ_HOME, exist_ok=True)

    targets = {
        BOLTZ_CHECKPOINT: os.path.join(_BOLTZ_HOME, "boltz1_conf.ckpt"),
        BOLTZ_CCD:        os.path.join(_BOLTZ_HOME, "ccd.pkl"),
    }

    all_ok = True
    for src, dst in targets.items():
        if os.path.isfile(dst):
            print(f"  ✅ Cache already present: {os.path.basename(dst)}")
            continue
        if not os.path.isfile(src):
            print(f"  ❌ Source not found: {src}")
            all_ok = False
            continue
        shutil.copy2(src, dst)
        sz = os.path.getsize(dst) / 1024**2
        print(f"  ✅ Cached {os.path.basename(dst)}  ({sz:.0f} MB)")

    return all_ok

_boltz_cache_ready = setup_boltz_cache()
print(f"\nBoltz-1 cache: {'✅ Ready' if _boltz_cache_ready else '⚠️  Partial — check paths'}")


# ── C1' extractor from mmCIF / PDB ───────────────────────────────────────────
def _extract_c1_from_file(filepath: str) -> "np.ndarray | None":
    """
    Parse a Boltz output file (.cif or .pdb) and extract C1' backbone atoms.

    Returns float32 array of shape (N_residues, 3), or None on parse failure.
    """
    if filepath.endswith(".cif"):
        try:
            import biotite.structure.io.pdbx as pdbx
            import biotite.structure as struc
            cf = pdbx.CIFFile.read(filepath)
            atom_arr = pdbx.get_structure(cf, model=1)
            c1_mask = (atom_arr.atom_name == "C1'")
            if c1_mask.sum() == 0:
                # Some CIFs label it C1* — try that
                c1_mask = (atom_arr.atom_name == "C1*")
            if c1_mask.sum() > 0:
                return atom_arr[c1_mask].coord.astype(np.float32)
        except Exception as _e:
            print(f"    [Boltz] CIF parse failed ({_e}), trying PDB fallback")

    # PDB fallback (or explicit .pdb)
    try:
        from Bio.PDB import PDBParser
        parser = PDBParser(QUIET=True)
        struct = parser.get_structure("boltz", filepath)
        coords = []
        for model in struct:
            for chain in model:
                for residue in chain:
                    atom_name = "C1'" if "C1'" in residue else ("C1*" if "C1*" in residue else None)
                    if atom_name:
                        coords.append(residue[atom_name].get_coord())
        if coords:
            return np.array(coords, dtype=np.float32)
    except Exception as _e:
        print(f"    [Boltz] PDB parse also failed: {_e}")

    return None


# ── Boltz-1 seed generator ────────────────────────────────────────────────────
# ── Boltz-1 seed generator ────────────────────────────────────────────────────
def generate_boltz_seed(target_id: str,
                         sequence: str,
                         out_dir: str,
                         device: str = None) -> "np.ndarray | None":
    """
    Generate a single 3D RNA structure with Boltz-1 using YAML input.
    """
    if device is None:
        device = DEVICE_BOLTZ

    if len(sequence) >= 800:
        print(f"  [Boltz] {target_id}: {len(sequence)} nt ≥ 800 — skipped (OOM guard)")
        return None

    if not _boltz_cache_ready:
        print(f"  [Boltz] Cache not ready — skipping {target_id}")
        return None

    # 1. Write YAML (Forces Boltz to recognize it as RNA and skip MSAs)
    target_out = os.path.join(out_dir, target_id)
    os.makedirs(target_out, exist_ok=True)
    yaml_path = os.path.join(target_out, f"{target_id}.yaml")
    
    yaml_content = f"""version: 1
sequences:
  - rna:
      id: A
      sequence: {sequence}
"""
    with open(yaml_path, "w") as _f:
        _f.write(yaml_content)

    # 2. Run Boltz via subprocess
    gpu_id = device.replace("cuda:", "") if "cuda:" in device else "0"
    
    _boltz_exe = os.path.join(os.path.dirname(sys.executable), "boltz")
    if not os.path.isfile(_boltz_exe):
        _boltz_exe = "boltz"   # fall back to PATH lookup

    cmd = [
        _boltz_exe, "predict", yaml_path,
        "--out_dir",               target_out,
        "--cache",                 _BOLTZ_HOME,
        "--accelerator",           "gpu",
        "--devices",               gpu_id,
        "--recycling_steps",       "3",
        "--sampling_steps",        "200",
        "--output_format",         "mmcif",
        "--diffusion_samples",     "1",
        "--override"
    ]
    
    print(f"  [Boltz] Running: {target_id} ({len(sequence)} nt) on {device} ...")
    try:
        result = subprocess.run(
            cmd,
            capture_output=True, text=True, timeout=600
        )
        if result.returncode != 0:
            print(f"  [Boltz] ❌ Non-zero exit: {result.stderr[-500:]}")
            return None
    except subprocess.TimeoutExpired:
        print(f"  [Boltz] ⏰ Timeout after 600 s — skipping {target_id}")
        return None
    except Exception as _e:
        print(f"  [Boltz] ❌ Subprocess error: {_e}")
        return None

    # 3. Locate output file
    _patterns = [
        os.path.join(target_out, "**", "*.cif"),
        os.path.join(target_out, "**", "*.pdb"),
    ]
    output_file = None
    for _pat in _patterns:
        _hits = sorted(glob.glob(_pat, recursive=True))
        if _hits:
            output_file = _hits[0]
            break

    if output_file is None:
        print(f"  [Boltz] ❌ No output file found under {target_out}")
        return None

    # 4. Extract C1' coordinates
    coords = _extract_c1_from_file(output_file)
    if coords is None or len(coords) == 0:
        print(f"  [Boltz] ❌ Could not extract C1' from {output_file}")
        return None

    print(f"  [Boltz] ✅ {target_id}: extracted {len(coords)} C1' atoms from {os.path.basename(output_file)}")
    return coords

# ── Boltz-1 batch seed generator (Phase 2a ensemble) ───────────────────────────────────
# ── Boltz-1 batch seed generator (Phase 2a ensemble) ───────────────────────────────────
def generate_boltz_seeds_batch(
    target_id: str,
    sequence: str,
    n_seeds: int,
    out_dir: str,
    device: str = None,
) -> "list[np.ndarray | None]":
    """
    Generate n_seeds independent 3D RNA structures with Boltz-1 using YAML input.
    """
    if device is None:
        device = DEVICE_BOLTZ

    if len(sequence) >= BOLTZ_MAX_LEN:
        print(f"  [Boltz batch] {target_id}: {len(sequence)} nt >= {BOLTZ_MAX_LEN} "
              f"-- OOM guard, skipping all {n_seeds} seeds")
        return [None] * n_seeds

    if not _boltz_cache_ready:
        print(f"  [Boltz batch] {target_id}: cache not ready -- skipping")
        return [None] * n_seeds

    gpu_id = device.replace("cuda:", "") if "cuda:" in device else "0"
    results = []

    for seed_i in range(n_seeds):
        seed_out  = os.path.join(out_dir, f"seed_{seed_i}")
        os.makedirs(seed_out, exist_ok=True)
        seed_tid   = f"{target_id}_b{seed_i}"
        yaml_path = os.path.join(seed_out, f"{seed_tid}.yaml")
        
        # Write YAML to strictly define entity as RNA
        yaml_content = f"""version: 1
sequences:
  - rna:
      id: A
      sequence: {sequence}
"""
        with open(yaml_path, "w") as _fh:
            _fh.write(yaml_content)

        _boltz_exe = os.path.join(os.path.dirname(sys.executable), "boltz")
        if not os.path.isfile(_boltz_exe):
            _boltz_exe = "boltz"
            
        cmd = [
            _boltz_exe, "predict", yaml_path,
            "--out_dir",               seed_out,
            "--cache",                 _BOLTZ_HOME,
            "--accelerator",           "gpu",
            "--devices",               gpu_id,
            "--recycling_steps",       "3",
            "--sampling_steps",        "200",
            "--output_format",         "mmcif",
            "--diffusion_samples",     "1",
            "--override"  
        ]
        
        print(f"  [Boltz] {target_id} seed {seed_i + 1}/{n_seeds} on {device} ({len(sequence)} nt) ...")
        
        try:
            result = subprocess.run(
                cmd, capture_output=True, text=True, timeout=600
            )
            if result.returncode != 0:
                print(f"  [Boltz] ❌ seed {seed_i}: exit {result.returncode} -- {result.stderr[-400:]}")
                results.append(None)
                continue
        except subprocess.TimeoutExpired:
            print(f"  [Boltz] ⏰ seed {seed_i}: timed out (600 s)")
            results.append(None)
            continue
        except Exception as _exc:
            print(f"  [Boltz] ❌ seed {seed_i}: subprocess error -- {_exc}")
            results.append(None)
            continue

        output_file = None
        for _pat in [
            os.path.join(seed_out, "**", "*.cif"),
            os.path.join(seed_out, "**", "*.pdb"),
        ]:
            _hits = sorted(glob.glob(_pat, recursive=True))
            if _hits:
                output_file = _hits[0]
                break

        if output_file is None:
            print(f"  [Boltz] ❌ seed {seed_i}: no output file found under {seed_out}")
            results.append(None)
            continue

        coords = _extract_c1_from_file(output_file)
        if coords is None or len(coords) == 0:
            print(f"  [Boltz] ❌ seed {seed_i}: C1' extraction failed from {os.path.basename(output_file)}")
            results.append(None)
        else:
            print(f"  [Boltz] ✅ seed {seed_i}: {len(coords)} C1' atoms from {os.path.basename(output_file)}")
            results.append(coords.astype(np.float32))

    n_ok = sum(1 for r in results if r is not None)
    print(f"  [Boltz batch] {target_id}: {n_ok}/{n_seeds} seeds collected")
    return results

print("✅ Boltz-1 wrapper ready.")

  ✅ Cached boltz1_conf.ckpt  (3429 MB)
  ✅ Cached ccd.pkl  (330 MB)

Boltz-1 cache: ✅ Ready
✅ Boltz-1 wrapper ready.


In [17]:
"""
overlap_stitcher — 3D Overlap Stitching for Megascale RNA targets.

Strategy:
  1. Slice the sequence into overlapping windows (stride = window - overlap).
  2. Predict local 3D coordinates for each window via a caller-supplied predictor_fn.
  3. Kabsch-align each new chunk onto the assembled global array using the
     shared overlap region as anchor atoms.
  4. Blend the seam with a linear taper to avoid discontinuous bonds.
  5. Concatenate the non-overlapping tail and return the full (L, 3) array.

Pure NumPy/SciPy CPU logic — never touches VRAM.
"""

import numpy as np
import warnings
from typing import Callable


# ---------------------------------------------------------------------------
# A-Form Helix Fallback Generator
# ---------------------------------------------------------------------------

def _aform_helix(n_nt: int) -> np.ndarray:
    """
    Idealised A-form RNA C1' backbone helix (fallback for failed predictor calls).
      Rise / residue : 2.81 A
      Twist / residue: 32.7 degrees
      Radius         : 9.0 A
    Returns (n_nt, 3) float32.
    """
    rise  = np.float32(2.81)
    twist = np.deg2rad(np.float32(32.7))
    r     = np.float32(9.0)
    indices = np.arange(n_nt, dtype=np.float32)
    x = r * np.cos(twist * indices)
    y = r * np.sin(twist * indices)
    z = rise * indices
    return np.stack([x, y, z], axis=-1)


# ---------------------------------------------------------------------------
# Kabsch Algorithm
# ---------------------------------------------------------------------------

def kabsch_align(mobile: np.ndarray, target: np.ndarray) -> tuple:
    """
    Optimal rotation R and translation T that superimposes `mobile` onto `target`
    in the least-squares sense (Kabsch 1976). Both arrays must have shape (M, 3),
    M >= 3.

    Returns
    -------
    R : (3, 3) float32 -- rotation matrix (proper, det = +1, chiral guard active).
    T : (3,)   float32 -- translation vector.
    """
    if mobile.shape != target.shape:
        raise ValueError(
            f"kabsch_align: shape mismatch -- mobile {mobile.shape} vs target {target.shape}"
        )
    if mobile.shape[0] < 3:
        raise ValueError("kabsch_align: need at least 3 anchor atoms.")

    # float64 internally to avoid catastrophic cancellation in SVD
    mob = mobile.astype(np.float64)
    tgt = target.astype(np.float64)
    mob_centroid = mob.mean(axis=0)
    tgt_centroid = tgt.mean(axis=0)
    H = (mob - mob_centroid).T @ (tgt - tgt_centroid)
    U, S, Vt = np.linalg.svd(H)

    # Reflection guard: ensures det(R) == +1 (proper rotation, chiral integrity)
    d = np.linalg.det(Vt.T @ U.T)
    D = np.eye(3, dtype=np.float64)
    D[2, 2] = np.sign(d)

    R = (Vt.T @ D @ U.T).astype(np.float32)
    T = (tgt_centroid - R.astype(np.float64) @ mob_centroid).astype(np.float32)
    return R, T


def apply_rigid_transform(coords: np.ndarray, R: np.ndarray, T: np.ndarray) -> np.ndarray:
    """Apply rotation R and translation T to an (N, 3) coordinate array."""
    return (R @ coords.T).T + T


# ---------------------------------------------------------------------------
# Linear Taper Blend
# ---------------------------------------------------------------------------

def _linear_taper_blend(prev_coords: np.ndarray, new_coords: np.ndarray) -> np.ndarray:
    """
    Blend two (overlap, 3) coordinate arrays with a linear weight ramp.
    w ramps 0 -> 1: i=0 is 100% prev, i=overlap-1 is 100% new.
    Spreads seam error across overlap atoms, preventing harmonic-force spikes
    in the JAX SHR integrator.
    """
    overlap = prev_coords.shape[0]
    w = np.linspace(0.0, 1.0, overlap, dtype=np.float32).reshape(-1, 1)
    return ((1.0 - w) * prev_coords + w * new_coords).astype(np.float32)


# ---------------------------------------------------------------------------
# Main Stitching Function
# ---------------------------------------------------------------------------

def stitch_megascale_rna(
    sequence: str,
    predictor_fn: Callable,
    window_size: int = 512,
    overlap: int = 128,
    verbose: bool = True,
) -> np.ndarray:
    """
    Predict and stitch 3D C1' backbone coordinates for a megascale RNA target.

    Parameters
    ----------
    sequence     : Full RNA sequence (length L).
    predictor_fn : str -> np.ndarray (N, 3). Called with each sub-sequence.
                   Must return float32 (N, 3). Falls back to A-form on failure.
    window_size  : Max subsequence length the predictor can handle.
    overlap      : Residues shared between consecutive windows (< window_size).
    verbose      : Print progress.

    Returns
    -------
    global_coords : np.ndarray (L, 3) float32.
    """
    L = len(sequence)
    if not (0 < overlap < window_size):
        raise ValueError(
            f"overlap ({overlap}) must satisfy 0 < overlap < window_size ({window_size})."
        )
    if L == 0:
        raise ValueError("sequence must be non-empty.")

    stride = window_size - overlap

    windows = []
    start = 0
    while start < L:
        end = min(start + window_size, L)
        windows.append((start, end))
        if end == L:
            break
        start += stride
    n_windows = len(windows)

    if verbose:
        print(
            f"[Stitcher] L={L} nt | window={window_size} | overlap={overlap} | "
            f"stride={stride} | chunks={n_windows}"
        )

    if n_windows == 1:
        if verbose:
            print("[Stitcher] Single chunk -- direct prediction, no stitching.")
        return _safe_predict(predictor_fn, sequence, 0, n_windows, verbose).astype(np.float32)

    s0, e0 = windows[0]
    global_coords = np.zeros((L, 3), dtype=np.float32)
    global_coords[s0:e0] = _safe_predict(predictor_fn, sequence[s0:e0], 0, n_windows, verbose)
    filled_up_to = e0

    for chunk_idx in range(1, n_windows):
        s_new, e_new = windows[chunk_idx]
        chunk_seq = sequence[s_new:e_new]
        chunk_len = e_new - s_new

        if verbose:
            print(f"[Stitcher] Chunk {chunk_idx+1}/{n_windows}  seq[{s_new}:{e_new}]  ({chunk_len} nt)")

        new_coords = _safe_predict(predictor_fn, chunk_seq, chunk_idx, n_windows, verbose)
        actual_overlap = min(overlap, chunk_len, filled_up_to - s_new)

        if actual_overlap < 3:
            warnings.warn(
                f"Chunk {chunk_idx}: only {actual_overlap} anchor atoms -- "
                "skipping Kabsch alignment.",
                RuntimeWarning,
            )
            R = np.eye(3, dtype=np.float32)
            T = np.zeros(3, dtype=np.float32)
        else:
            mobile_anchor = new_coords[:actual_overlap].copy()
            target_anchor = global_coords[s_new : s_new + actual_overlap].copy()
            try:
                R, T = kabsch_align(mobile_anchor, target_anchor)
            except (ValueError, np.linalg.LinAlgError) as exc:
                warnings.warn(
                    f"Chunk {chunk_idx}: Kabsch SVD failed ({exc}). Using identity transform.",
                    RuntimeWarning,
                )
                R = np.eye(3, dtype=np.float32)
                T = np.zeros(3, dtype=np.float32)

        new_coords_aligned = apply_rigid_transform(new_coords, R, T)
        prev_seam = global_coords[s_new : s_new + actual_overlap].copy()
        new_seam  = new_coords_aligned[:actual_overlap]
        global_coords[s_new : s_new + actual_overlap] = _linear_taper_blend(prev_seam, new_seam)
        global_coords[s_new + actual_overlap : e_new] = new_coords_aligned[actual_overlap:]
        filled_up_to = e_new

    if verbose:
        print(f"[Stitcher] Assembly complete. Final shape: {global_coords.shape}")
    return global_coords


# ---------------------------------------------------------------------------
# Internal helper: safe predictor call with A-form fallback
# ---------------------------------------------------------------------------

def _safe_predict(
    predictor_fn: Callable,
    sub_seq: str,
    chunk_idx: int,
    n_chunks: int,
    verbose: bool,
) -> np.ndarray:
    """Call predictor_fn; fall back to A-form helix on any failure."""
    n = len(sub_seq)
    try:
        coords = predictor_fn(sub_seq)
        coords = np.asarray(coords, dtype=np.float32)
        if coords.shape != (n, 3):
            raise ValueError(f"predictor_fn returned shape {coords.shape}; expected ({n}, 3).")
        if not np.all(np.isfinite(coords)):
            raise ValueError("predictor_fn returned non-finite coordinates.")
        if verbose:
            print(
                f"  [Predictor] chunk {chunk_idx+1}/{n_chunks}: "
                f"OK  ({n} nt, max_coord={np.abs(coords).max():.2f} A)"
            )
        return coords
    except Exception as exc:
        warnings.warn(
            f"predictor_fn failed on chunk {chunk_idx+1}/{n_chunks} "
            f"(seq len={n}): {exc!r}\n"
            "  -> Falling back to A-form helix for this chunk.",
            RuntimeWarning,
            stacklevel=3,
        )
        return _aform_helix(n)


print("✅ stitch_megascale_rna module loaded.")


✅ stitch_megascale_rna module loaded.


## RNA-FM HWS (Hierarchical Windowed Sensor) Safe Loader

Handles the 1,024-nt RNA-FM truncation limit using sliding windows with 50% overlap
and linear taper blending. For megascale targets (9MME = 4,640 nt), extracts embeddings
across the full sequence and stitches them into a global contact map C_ij.


In [18]:
# ============================================================
# LAYER: RNA-FM HWS (Hierarchical Windowed Sensor) Safe Loader
#
# Handles the 1,024-nt RNA-FM truncation limit using sliding
# windows with adaptive overlap and linear taper blending.
#
# For sequences <= 1022 nt: standard single-pass embedding
# For sequences >  1022 nt: HWS windowed extraction + stitching
#
# The stitched embeddings are converted to a global contact map
# (C_ij) that feeds the Hamiltonian E_DL restraint term.
# ============================================================

import torch
import sys
import types
import numpy as np
from scipy.fft import dct, idct

# ==========================================
# 1. RNA-FM EMBEDDING CONFIG (1D LM)
# ==========================================
HWS_WINDOW_SIZE = 1022   # Max allowed by RNA-FM architecture
HWS_STRIDE      = 768    # 254 nt overlap
HWS_EMB_DIM     = 640
HWS_TAPER_LEN   = 128

RNA_FM_LOADED = False
RNA_FM_MODEL  = None
RNA_FM_ALPHABET = None

# ==========================================
# 2. PROTENIX FOLDING CONFIG (3D AF3-Clone)
# ==========================================
MODEL_NAME    = "protenix_base_20250630_v1.0.0"
N_SAMPLE      = 5        # 5 diverse PDBs for the Kaggle grader
SEED          = 42
MAX_SEQ_LEN   = 512      # VRAM safety limit for 3D generation
CHUNK_OVERLAP = 128      # Overlap for stitching the 3D chunks

# Environment Settings
USE_MSA      = "false"
USE_TEMPLATE = "false"
USE_RNA_MSA  = "true"

# ==========================================
# 3. ADAPTIVE ROUTING LOGIC
# ==========================================
USE_PROTENIX   = True
MIN_SIMILARITY = 0.0

def get_adaptive_identity(seq_len):
    """Dynamically lowers the required template identity for massive targets."""
    if seq_len > 3000:
        return 20.0  # We will take whatever hints we can get for massive RNAs
    elif seq_len > 1000:
        return 35.0
    else:
        return 50.0  # Strict for small, easy RNAs

# We will call this function later once we load the specific CSV sequence:
# adaptive_identity_threshold = get_adaptive_identity(len(current_sequence))

def load_rna_fm_once(device=None, weight_path=None):
    """
    Load RNA-FM model once and cache globally.
    Handles multi-GPU device routing and offline weight loading.
    """
    global RNA_FM_LOADED, RNA_FM_MODEL, RNA_FM_ALPHABET
    
    # 1. Return cached model if it's already loaded
    if RNA_FM_LOADED and RNA_FM_MODEL is not None:
        return RNA_FM_MODEL, RNA_FM_ALPHABET

    # 2. Resolve Device: Respect the 'device' argument passed by main()
    if device is None:
        device = "cuda:0" if torch.cuda.is_available() else "cpu"
        
    # 3. Resolve Weights Path
    if weight_path is None:
        # Falls back to a global variable if you defined one, or use the direct path
        weight_path = globals().get(
            "RNA_FM_WEIGHTS", 
            "/kaggle/input/datasets/aarryyaannnnnn/rna-fm-weights/rna-fm-weights/RNA-FM_pretrained.pth"
        )

    print(f"Loading RNA-FM to {device}...")
    
    try:
        # 4. Safety Net: Mock ptflops inside the loader so it never crashes
        if 'ptflops' not in sys.modules:
            mock_ptflops = types.ModuleType('ptflops')
            mock_ptflops.get_model_complexity_info = lambda *args, **kwargs: (0, 0)
            sys.modules['ptflops'] = mock_ptflops

        import fm
        
        # 5. Load the model from offline weights
        model, alphabet = fm.pretrained.rna_fm_t12(weight_path)
        
        # 6. Move to the correct GPU (cuda:0 or cuda:1) and set to evaluation mode
        model = model.eval().to(device)
        
        # 7. Cache in global memory
        RNA_FM_MODEL = model
        RNA_FM_ALPHABET = alphabet
        RNA_FM_LOADED = True
        
        print(f"✅ RNA-FM loaded successfully to [{device}]")
        return model, alphabet
        
    except Exception as e:
        print(f"❌ RNA-FM loading failed: {e}")
        RNA_FM_LOADED = True   # Prevent infinite retry loops on failure
        return None, None


def get_rna_fm_embeddings_safe(sequence, model=None, alphabet=None):
    """
    HWS Safe Loader for RNA-FM embeddings.

    For sequences <= 1022 nt: single-pass through RNA-FM.
    For sequences > 1022 nt (e.g. 9MME = 4,640 nt):
        Sliding window with 512-nt stride and linear taper blending.
        Overlapping regions are averaged via weighted accumulation
        to prevent hard boundary artifacts.

    Returns: (N, 640) numpy array, or None if RNA-FM unavailable.
    """
    if model is None or alphabet is None:
        model, alphabet = load_rna_fm_once()
    if model is None:
        return None

    device = next(model.parameters()).device
    N = len(sequence)
    batch_converter = alphabet.get_batch_converter()

    if N <= HWS_WINDOW_SIZE:
        # Standard single pass
        data = [("target", sequence)]
        _, _, batch_tokens = batch_converter(data)
        with torch.no_grad():
            results = model(batch_tokens.to(device), repr_layers=[12])
        emb = results["representations"][12][0, 1:N+1].cpu().numpy()
        del batch_tokens, results
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return emb

    # HWS windowed extraction
    print(f"  [HWS] Megascale: {N} nt, window={HWS_WINDOW_SIZE}, stride={HWS_STRIDE}")
    emb_accum    = np.zeros((N, HWS_EMB_DIM), dtype=np.float64)
    weight_accum = np.zeros((N, 1),            dtype=np.float64)
    n_windows = 0

    for start in range(0, N, HWS_STRIDE):
        end = min(start + HWS_WINDOW_SIZE, N)
        win_len = end - start
        if win_len < 32:
            break

        sub_seq = sequence[start:end]
        try:
            data = [("window", sub_seq)]
            _, _, batch_tokens = batch_converter(data)
            with torch.no_grad():
                results = model(batch_tokens.to(device), repr_layers=[12])
            sub_emb = results["representations"][12][0, 1:win_len+1].cpu().numpy()

            # Linear taper weights
            taper = np.ones(win_len, dtype=np.float64)
            taper_len = min(HWS_TAPER_LEN, win_len // 4)
            if start > 0 and taper_len > 0:
                taper[:taper_len] = np.linspace(0.0, 1.0, taper_len)
            if end < N and taper_len > 0:
                taper[-taper_len:] = np.linspace(1.0, 0.0, taper_len)

            emb_accum[start:end]    += sub_emb * taper[:, None]
            weight_accum[start:end] += taper[:, None]
            n_windows += 1

            del batch_tokens, results, sub_emb
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        except Exception as e:
            print(f"    [HWS] Window [{start}:{end}] failed: {e}")
            weight_accum[start:end] += 1e-8
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        if end >= N:
            break

    weight_accum = np.maximum(weight_accum, 1e-8)
    blended = (emb_accum / weight_accum).astype(np.float32)
    blended = np.nan_to_num(blended, nan=0.0, posinf=0.0, neginf=0.0)
    print(f"  [HWS] {n_windows} windows -> ({N}, {HWS_EMB_DIM}) embeddings")
    return blended


def embeddings_to_contact_map(embeddings, threshold=0.85, min_seq_sep=6, chunk_size=500):
    """Convert RNA-FM embeddings to a global pairwise binary contact map via cosine similarity."""
    if embeddings is None:
        return None
    N = embeddings.shape[0]
    norms = np.sqrt(np.sum(embeddings**2, axis=1, keepdims=True) + 1e-8)
    normed = embeddings / norms

    if N <= 2000:
        sim = normed @ normed.T
    else:
        sim = np.zeros((N, N), dtype=np.float32)
        for i in range(0, N, chunk_size):
            ie = min(i + chunk_size, N)
            sim[i:ie, :] = (normed[i:ie] @ normed.T).astype(np.float32)

    contact_prob = (sim + 1.0) / 2.0
    for k in range(-min_seq_sep, min_seq_sep + 1):
        np.fill_diagonal(contact_prob[max(0, -k):, max(0, k):], 0.0)

    contact_map = (contact_prob > threshold).astype(np.float32)
    contact_map = np.maximum(contact_map, contact_map.T)

    n_contacts = int(np.sum(np.triu(contact_map, k=1)))
    print(f"  [HWS] Contact map: {N}x{N}, {n_contacts} contacts (threshold={threshold})")
    return contact_map


def get_contact_map_for_sequence(sequence, threshold=0.85):
    """Full HWS pipeline: sequence -> windowed embeddings -> global contact map.
    Returns (N, N) float32 contact map, or None if RNA-FM unavailable."""
    embeddings = get_rna_fm_embeddings_safe(sequence)
    return embeddings_to_contact_map(embeddings, threshold=threshold)


def extract_contacts_from_template(template_coords, d_cutoff=12.0, min_sep=6):
    """Extract contact pairs from a template structure (fallback when RNA-FM unavailable)."""
    N = len(template_coords)
    if N > 2000:
        idx = np.linspace(0, N-1, min(N, 800)).astype(int)
        sub = template_coords[idx]
        diff = sub[:, None, :] - sub[None, :, :]
        dists = np.sqrt(np.sum(diff**2, axis=-1))
        sep = np.abs(idx[:, None] - idx[None, :])
        mask = (sep >= min_sep) & (dists < d_cutoff) & np.triu(np.ones_like(dists, dtype=bool), k=1)
        ii, jj = np.where(mask)
        pairs = np.stack([idx[ii], idx[jj]], axis=1)
    else:
        diff = template_coords[:, None, :] - template_coords[None, :, :]
        dists = np.sqrt(np.sum(diff**2, axis=-1))
        sep_mat = np.abs(np.arange(N)[:, None] - np.arange(N)[None, :])
        mask = (sep_mat >= min_sep) & (dists < d_cutoff) & np.triu(np.ones((N, N), dtype=bool), k=1)
        ii, jj = np.where(mask)
        pairs = np.stack([ii, jj], axis=1) if len(ii) > 0 else np.zeros((0, 2), dtype=np.int64)

    # Convert pairs to dense contact map
    contact_map = np.zeros((N, N), dtype=np.float32)
    if len(pairs) > 0:
        contact_map[pairs[:, 0], pairs[:, 1]] = 1.0
        contact_map[pairs[:, 1], pairs[:, 0]] = 1.0
    return contact_map


print("HWS Safe Loader initialized")


HWS Safe Loader initialized


## SHR Physics Engine (Stochastic Hamiltonian Relaxation)

Differentiable physics from Manuscript V8.0. Operates on C1' coordinates.

Hamiltonian: H(x) = E_bond + E_rep + E_DL + E_Rg
- E_bond: Harmonic backbone bonds (d0=5.95A for C1'-C1')
- E_rep: Pairwise steric repulsion (sigma=3.0A)
- E_DL: RNA-FM contact restraints via HWS pipeline
- E_Rg: Radius of gyration restraint (Flory scaling)

Sobolev H1 gradient preconditioning via DCT-II spectral filtering.


## Baseline TBM Core Functions

Adaptive Template-Based Modeling from the 0.415 baseline.
Includes: sequence alignment, template adaptation, diversity transforms,
chain segment handling, de-novo A-form helix generation.


In [19]:
# ─────────────── Paths & Constants ───────────────────────────────────────────
DATA_BASE              = "/kaggle/input/stanford-rna-3d-folding-2"
DEFAULT_TEST_CSV       = f"{DATA_BASE}/test_sequences.csv"
DEFAULT_TRAIN_CSV      = f"{DATA_BASE}/train_sequences.csv"
DEFAULT_OUTPUT         = "/kaggle/working/submission.csv"

# This must point to the folder ABOVE 'checkpoint'
DEFAULT_CODE_DIR = "/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1"
DEFAULT_ROOT_DIR = DEFAULT_CODE_DIR

MODEL_N_SAMPLE = N_SAMPLE


# ─────────────── General Utilities ───────────────────────────────────────────
def seed_everything(seed: int) -> None:
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.enabled = True
    torch.use_deterministic_algorithms(True)


def resolve_paths():
    test_csv   = os.environ.get("TEST_CSV",           DEFAULT_TEST_CSV)
    output_csv = os.environ.get("SUBMISSION_CSV",     DEFAULT_OUTPUT)
    code_dir   = os.environ.get("PROTENIX_CODE_DIR",  DEFAULT_CODE_DIR)
    root_dir   = os.environ.get("PROTENIX_ROOT_DIR",  DEFAULT_ROOT_DIR)
    return test_csv, output_csv, code_dir, root_dir


def ensure_required_files(root_dir: str) -> None:
    for p, name in [
        (Path(root_dir) / "checkpoint" / f"{MODEL_NAME}.pt",          "checkpoint"),
        (Path(root_dir) / "common" / "components.cif",                "CCD file"),
        (Path(root_dir) / "common" / "components.cif.rdkit_mol.pkl",  "CCD cache"),
    ]:
        if not p.exists():
            raise FileNotFoundError(f"Missing {name}: {p}")


# ─────────────── Protenix Input / Config Helpers ─────────────────────────────
def build_input_json(df: pd.DataFrame, json_path: str) -> None:
    data = [
        {
            "name": row["target_id"],
            "covalent_bonds": [],
            "sequences": [{"rnaSequence": {"sequence": row["sequence"], "count": 1}}],
        }
        for _, row in df.iterrows()
    ]
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(data, f)


def build_configs(input_json_path: str, dump_dir: str, model_name: str):
    from configs.configs_base import configs as configs_base
    from configs.configs_data import data_configs
    from configs.configs_inference import inference_configs
    from configs.configs_model_type import model_configs
    from protenix.config.config import parse_configs

    base = {**configs_base, **{"data": data_configs}, **inference_configs}

    def deep_update(t, p):
        for k, v in p.items():
            if isinstance(v, dict) and k in t and isinstance(t[k], dict):
                deep_update(t[k], v)
            else:
                t[k] = v

    deep_update(base, model_configs[model_name])
    arg_str = " ".join([
        f"--model_name {model_name}",
        f"--input_json_path {input_json_path}",
        f"--dump_dir {dump_dir}",
        f"--use_msa {USE_MSA}",
        f"--use_template {USE_TEMPLATE}",
        f"--use_rna_msa {USE_RNA_MSA}",
        f"--sample_diffusion.N_sample {MODEL_N_SAMPLE}",
        f"--seeds {SEED}",
    ])
    return parse_configs(configs=base, arg_str=arg_str, fill_required_with_null=True)


def get_c1_mask(data: dict, atom_array) -> torch.Tensor:
    # 1. Try atom_array attributes first
    if atom_array is not None:
        try:
            if hasattr(atom_array, "centre_atom_mask"):
                m = atom_array.centre_atom_mask == 1
                if hasattr(atom_array, "is_rna"):
                    m = m & atom_array.is_rna
                return torch.from_numpy(m).bool()

            if hasattr(atom_array, "atom_name"):
                base = atom_array.atom_name == "C1'"
                if hasattr(atom_array, "is_rna"):
                    base = base & atom_array.is_rna
                return torch.from_numpy(base).bool()
        except Exception:
            pass

    # 2. Fallback to feature dict
    f = data["input_feature_dict"]

    if "center_atom_mask" in f:
        return (f["center_atom_mask"] == 1).bool()
    if "centre_atom_mask" in f:
        return (f["centre_atom_mask"] == 1).bool()

    if "atom_name" in f:
        pass

    return (f["atom_to_tokatom_idx"] == 11).bool()


def get_feature_c1_mask(data: dict) -> torch.Tensor:
    f = data["input_feature_dict"]
    if "centre_atom_mask" in f:
        return f["centre_atom_mask"].long() == 1
    return f["atom_to_tokatom_idx"].long() == 12


def coords_to_rows(target_id: str, seq: str, coords: np.ndarray) -> list:
    """coords shape: (N_SAMPLE, seq_len, 3)"""
    rows = []
    for i in range(len(seq)):
        row = {"ID": f"{target_id}_{i + 1}", "resname": seq[i], "resid": i + 1}
        for s in range(N_SAMPLE):
            if s < coords.shape[0] and i < coords.shape[1]:
                x, y, z = coords[s, i]
            else:
                x, y, z = 0.0, 0.0, 0.0
            row[f"x_{s + 1}"] = float(x)
            row[f"y_{s + 1}"] = float(y)
            row[f"z_{s + 1}"] = float(z)
        rows.append(row)
    return rows




# ─────────────────────────────────────────────────────────────────────────────
# v19 MEGASCALE ARMOR: Ensemble Alignment + Explosion Guard + Cross-Model Sanitization
# Fixes "Catastrophic Megascale Drift" from submission_44.
#
# Key invariants:
#   • Model 1 (Slot 0 = Crystal Guard) is NEVER modified.
#   • Models 2-5 are Kabsch-aligned onto Model 1 in float64.
#   • Exploded models (mean dist > 500 Å) are replaced with noisy Model 1.
#   • Cross-model sanitizer: if ANY model has a bad residue, ALL 5 get sentinel.
# ─────────────────────────────────────────────────────────────────────────────

SENTINEL               = -2_000_000.0
_EXPLOSION_THRESHOLD_A = 500.0
_NOISE_STD_A           = 0.5
_MIN_VALID_ATOMS       = 4


def _valid_coord_mask(coords: np.ndarray) -> np.ndarray:
    """True for atoms that are finite, non-sentinel, and non-zero-triplet."""
    finite       = np.isfinite(coords).all(axis=-1)
    non_sentinel = (coords > SENTINEL + 1.0).all(axis=-1)
    non_zero     = ~((coords[:, 0] == 0.0) & (coords[:, 1] == 0.0) & (coords[:, 2] == 0.0))
    return finite & non_sentinel & non_zero


def align_ensemble_to_slot0(models_list):
    """
    Kabsch-align Models 2-N onto Model 1 (Slot 0).
    Model 1 is NEVER modified (Crystal Guard invariant).
    Exploded models are replaced with noisy copies of Model 1.
    
    Args:
        models_list: list of np.ndarray, each (L, 3)
    Returns:
        list of np.ndarray, each (L, 3), aligned
    """
    n_models = len(models_list)
    aligned = [m.copy() for m in models_list]
    ref = aligned[0]   # READ-ONLY reference
    ref_mask = _valid_coord_mask(ref)

    if ref_mask.sum() < _MIN_VALID_ATOMS:
        return aligned

    rng = np.random.default_rng(seed=12345)

    for i in range(1, n_models):
        model_mask = _valid_coord_mask(aligned[i])
        shared = ref_mask & model_mask

        if shared.sum() < _MIN_VALID_ATOMS:
            print(f"    Model {i+1}: <{_MIN_VALID_ATOMS} shared atoms → noisy copy of Model 1")
            aligned[i] = ref.copy() + rng.normal(0, _NOISE_STD_A, ref.shape).astype(ref.dtype)
            continue

        # Kabsch in float64 for numerical stability on megascale targets
        mobile_pts = aligned[i][shared].astype(np.float64)
        target_pts = ref[shared].astype(np.float64)
        try:
            mob_centroid = mobile_pts.mean(axis=0)
            tgt_centroid = target_pts.mean(axis=0)
            H = (mobile_pts - mob_centroid).T @ (target_pts - tgt_centroid)
            U, _S, Vt = np.linalg.svd(H)
            d = np.linalg.det(Vt.T @ U.T)
            if d < 0:
                Vt[-1, :] *= -1
            R = Vt.T @ U.T
            full_centered = aligned[i].astype(np.float64) - mob_centroid
            aligned[i] = (full_centered @ R.T + tgt_centroid).astype(ref.dtype)
        except Exception:
            aligned[i] = ref.copy() + rng.normal(0, _NOISE_STD_A, ref.shape).astype(ref.dtype)
            continue

        # Post-alignment explosion check
        shared_after = ref_mask & _valid_coord_mask(aligned[i])
        if shared_after.sum() < _MIN_VALID_ATOMS:
            mean_dist = np.inf
        else:
            mean_dist = np.linalg.norm(
                aligned[i][shared_after] - ref[shared_after], axis=-1
            ).mean()

        if mean_dist > _EXPLOSION_THRESHOLD_A:
            print(f"    ⚠️  Model {i+1}: EXPLODED (mean dist={mean_dist:.1f} Å) "
                  f"→ noisy copy of Model 1")
            aligned[i] = ref.copy() + rng.normal(0, _NOISE_STD_A, ref.shape).astype(ref.dtype)

    return aligned


def sanitize_coordinates(models_list, max_abs=1499.0, sentinel=SENTINEL):
    """
    Cross-model coordinate sanitization (vectorized).
    
    If ANY model has a bad coordinate at residue i (non-finite, |x|>max_abs,
    or all-zero), ALL 5 models get the sentinel for residue i.
    This prevents the Kaggle grader from scoring one model's valid atom
    against another model's garbage at the same residue.
    
    Args:
        models_list: list of np.ndarray, each (L, 3)
    Returns:
        list of np.ndarray, each (L, 3), sanitized
    """
    n_models = len(models_list)
    ensemble = np.stack(models_list, axis=0).copy()  # (M, L, 3)

    not_finite       = ~np.isfinite(ensemble)                       # (M, L, 3)
    out_of_range     = np.abs(ensemble) > max_abs                   # (M, L, 3)
    bad_component    = not_finite | out_of_range                    # (M, L, 3)
    bad_residue_per_model = bad_component.any(axis=2)               # (M, L)
    bad_residue_any_model = bad_residue_per_model.any(axis=0)       # (L,)

    # Also flag all-zero triplets (unresolved stitcher padding)
    for m in range(n_models):
        all_zero = ((ensemble[m, :, 0] == 0.0) &
                    (ensemble[m, :, 1] == 0.0) &
                    (ensemble[m, :, 2] == 0.0))
        bad_residue_any_model |= all_zero

    n_bad = int(bad_residue_any_model.sum())
    if n_bad > 0:
        ensemble[:, bad_residue_any_model, :] = sentinel
        print(f"    Sanitized: {n_bad}/{ensemble.shape[1]} residues → sentinel across all models")

    return [ensemble[i] for i in range(n_models)]


def coords_to_rows_v18(target_id, seq, coords, n_sample=5):
    """Robust wide-format row builder with 3-decimal rounding.
    
    Accepts coords as either:
      - np.ndarray of shape (n_sample, L, 3)  [stacked]
      - list of np.ndarray each (L, 3)         [list form]
    """
    SENTINEL_VAL = -2_000_000.0
    if isinstance(coords, list):
        coords = np.stack(coords, axis=0)
    rows = []
    for i in range(len(seq)):
        row = {"ID": f"{target_id}_{i + 1}", "resname": seq[i], "resid": i + 1}
        for s in range(n_sample):
            if s < coords.shape[0] and i < coords.shape[1]:
                x, y, z = coords[s, i]
                row[f"x_{s+1}"] = round(float(x), 3)
                row[f"y_{s+1}"] = round(float(y), 3)
                row[f"z_{s+1}"] = round(float(z), 3)
            else:
                row[f"x_{s+1}"] = SENTINEL_VAL
                row[f"y_{s+1}"] = SENTINEL_VAL
                row[f"z_{s+1}"] = SENTINEL_VAL
        rows.append(row)
    return rows


def pad_samples(coords: np.ndarray, n: int) -> np.ndarray:
    if coords.shape[0] >= n:
        return coords[:n]
    if coords.shape[0] == 0:
        return np.zeros((n, coords.shape[1], 3), dtype=coords.dtype)
    extra = np.repeat(coords[:1], n - coords.shape[0], axis=0)
    return np.concatenate([coords, extra], axis=0)


# ─────────────── TBM Core Functions ──────────────────────────────────────────
def _make_aligner() -> PairwiseAligner:
    """
    Build a global PairwiseAligner with RNA-optimised gap penalties.

    Attribute names use the Biopython >= 1.82 API (canonicalised in 1.86):
      query_*  -> open/extend_{left,right}_deletion_score
      target_* -> open/extend_{left,right}_insertion_score
    Using the old names triggers BiopythonDeprecationWarning on every
    align() call (8 warnings x N sequences = log noise).
    """
    al = PairwiseAligner()
    al.mode             = "global"
    al.match_score      = 2
    al.mismatch_score   = -1.5
    al.open_gap_score   = -8
    al.extend_gap_score = -0.4

    # ── End-gap penalties (Biopython >= 1.82 canonical names) ────────────────
    # "deletion"  = gap in the target alignment  (query side opens/extends)
    # "insertion" = gap in the query alignment   (target side opens/extends)
    al.open_left_deletion_score     = -8
    al.extend_left_deletion_score   = -0.4
    al.open_right_deletion_score    = -8
    al.extend_right_deletion_score  = -0.4
    al.open_left_insertion_score    = -8
    al.extend_left_insertion_score  = -0.4
    al.open_right_insertion_score   = -8
    al.extend_right_insertion_score = -0.4
    return al


_aligner = _make_aligner()


def parse_stoichiometry(stoich: str) -> list:
    if pd.isna(stoich) or str(stoich).strip() == "":
        return []
    return [(ch.strip(), int(cnt)) for part in str(stoich).split(";")
            for ch, cnt in [part.split(":")]]


def parse_fasta(fasta_content: str) -> dict:
    out, cur, parts = {}, None, []
    for line in str(fasta_content).splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith(">"):
            if cur is not None:
                out[cur] = "".join(parts)
            cur = line[1:].split()[0]
            parts = []
        else:
            parts.append(line.replace(" ", ""))
    if cur is not None:
        out[cur] = "".join(parts)
    return out


def get_chain_segments(row) -> list:
    seq    = row["sequence"]
    stoich = row.get("stoichiometry", "")
    all_sq = row.get("all_sequences", "")
    if (pd.isna(stoich) or pd.isna(all_sq)
            or str(stoich).strip() == "" or str(all_sq).strip() == ""):
        return [(0, len(seq))]
    try:
        chain_dict = parse_fasta(all_sq)
        order = parse_stoichiometry(stoich)
        segs, pos = [], 0
        for ch, cnt in order:
            base = chain_dict.get(ch)
            if base is None:
                return [(0, len(seq))]
            for _ in range(cnt):
                segs.append((pos, pos + len(base)))
                pos += len(base)
        return segs if pos == len(seq) else [(0, len(seq))]
    except Exception:
        return [(0, len(seq))]


def build_segments_map(df: pd.DataFrame) -> tuple:
    seg_map, stoich_map = {}, {}
    for _, r in df.iterrows():
        tid               = r["target_id"]
        seg_map[tid]      = get_chain_segments(r)
        raw_s             = r.get("stoichiometry", "")
        stoich_map[tid]   = "" if pd.isna(raw_s) else str(raw_s)
    return seg_map, stoich_map


def process_labels(labels_df: pd.DataFrame) -> dict:
    coords = {}
    prefixes = labels_df["ID"].str.rsplit("_", n=1).str[0]
    for prefix, grp in labels_df.groupby(prefixes):
        coords[prefix] = grp.sort_values("resid")[["x_1", "y_1", "z_1"]].values
    return coords


def _build_aligned_strings(query_seq, template_seq, alignment):
    q_segs, t_segs = alignment.aligned
    aq, at, qi, ti = [], [], 0, 0
    for (qs, qe), (ts, te) in zip(q_segs, t_segs):
        while qi < qs: aq.append(query_seq[qi]);    at.append("-");              qi += 1
        while ti < ts: aq.append("-");              at.append(template_seq[ti]); ti += 1
        for qp, tp in zip(range(qs, qe), range(ts, te)):
            aq.append(query_seq[qp]); at.append(template_seq[tp])
        qi, ti = qe, te
    while qi < len(query_seq):    aq.append(query_seq[qi]);    at.append("-");              qi += 1
    while ti < len(template_seq): aq.append("-");              at.append(template_seq[ti]); ti += 1
    return "".join(aq), "".join(at)


def find_similar_sequences_detailed(query_seq, train_seqs_df, train_coords_dict, top_n=30):
    results = []
    for _, row in train_seqs_df.iterrows():
        tid, tseq = row["target_id"], row["sequence"]
        if tid not in train_coords_dict:
            continue
        if abs(len(tseq) - len(query_seq)) / max(len(tseq), len(query_seq)) > 0.3:
            continue
        aln       = next(iter(_aligner.align(query_seq, tseq)))
        norm_s    = aln.score / (2 * min(len(query_seq), len(tseq)))
        identical = sum(
            1 for (qs, qe), (ts, te) in zip(*aln.aligned)
            for qp, tp in zip(range(qs, qe), range(ts, te))
            if query_seq[qp] == tseq[tp]
        )
        pct_id = 100 * identical / len(query_seq)
        aq, at = _build_aligned_strings(query_seq, tseq, aln)
        results.append((tid, tseq, norm_s, train_coords_dict[tid], pct_id, aq, at))
    results.sort(key=lambda x: x[2], reverse=True)
    return results[:top_n]


def adapt_template_to_query(query_seq, template_seq, template_coords) -> np.ndarray:
    """Adapt template coordinates to query with compact gap filling.

    Gap-filling strategy:
      * Interpolated gap  (anchors on both sides) : linear lerp — unchanged.
      * Trailing gap      (left anchor only)      : compact local helix spiralling
        outward from the last known residue, z wrapped every 120 residues so
        the total z-extension stays under ~360 Å regardless of gap size.
      * Leading gap       (right anchor only)     : same spiral from first known.
      * Orphan            (no template anchor)    : bounded super-helix at origin.

    Old fallbacks:
      elif pv >= 0: new_coords[i] = new_coords[pv] + [3, 0, 0]   ← rod along x
      elif nv >= 0: new_coords[i] = new_coords[nv] + [3, 0, 0]   ← rod along x
      else:         new_coords[i] = [i * 3, 0, 0]                ← pure linear rod
    A 2,000-residue trailing gap produced a 6,000 Å rod → clipped to ±9,999.
    """
    aln        = next(iter(_aligner.align(query_seq, template_seq)))
    new_coords = np.full((len(query_seq), 3), np.nan)
    for (qs, qe), (ts, te) in zip(*aln.aligned):
        chunk = template_coords[ts:te]
        if len(chunk) == (qe - qs):
            new_coords[qs:qe] = chunk

    def _helix_offset(step: int) -> np.ndarray:
        """Compact spiral offset for `step` residues away from an anchor.

        Radius 5 Å, pitch 3 Å/residue, z wraps every 120 residues so the
        total z-extension never exceeds ~360 Å regardless of gap length.
        """
        ang = step * 0.6
        return np.array(
            [5.0 * np.cos(ang), 5.0 * np.sin(ang), (step % 120) * 3.0],
            dtype=np.float32,
        )

    for i in range(len(new_coords)):
        if not np.isnan(new_coords[i, 0]):
            continue
        pv = next((j for j in range(i - 1, -1, -1)
                   if not np.isnan(new_coords[j, 0])), -1)
        nv = next((j for j in range(i + 1, len(new_coords))
                   if not np.isnan(new_coords[j, 0])), -1)

        if pv >= 0 and nv >= 0:
            # Interpolated gap — linear lerp between the two anchors.
            w = (i - pv) / (nv - pv)
            new_coords[i] = (1 - w) * new_coords[pv] + w * new_coords[nv]

        elif pv >= 0:
            # Trailing gap — spiral outward from the last known residue.
            new_coords[i] = new_coords[pv] + _helix_offset(i - pv)

        elif nv >= 0:
            # Leading gap — spiral back from the first known residue.
            new_coords[i] = new_coords[nv] + _helix_offset(nv - i)

        else:
            # Orphan residue — no template anchor at all.
            # Use the same bounded super-helix math as generate_rna_structure.
            super_ang     = i * 0.05
            super_r       = 30.0 + 15.0 * np.sin(i * 0.01)
            ang           = i * 0.6
            new_coords[i] = np.array(
                [
                    super_r * np.cos(super_ang) + 10.0 * np.cos(ang),
                    super_r * np.sin(super_ang) + 10.0 * np.sin(ang),
                    (i % 400) * 1.5,
                ],
                dtype=np.float32,
            )

    return np.nan_to_num(new_coords)


def adaptive_rna_constraints(coords, target_id, segments_map, confidence=1.0, passes=2) -> np.ndarray:
    X        = coords.copy()
    segments = segments_map.get(target_id, [(0, len(X))])
    strength = max(0.75 * (1.0 - min(confidence, 0.97)), 0.02)
    for _ in range(passes):
        for s, e in segments:
            C = X[s:e]; L = e - s
            if L < 3:
                continue
            # bond i–i+1  ~5.95 Å
            d    = C[1:] - C[:-1]; dist = np.linalg.norm(d, axis=1) + 1e-6
            adj  = d * ((5.95 - dist) / dist)[:, None] * (0.22 * strength)
            C[:-1] -= adj; C[1:] += adj
            # soft i–i+2  ~10.2 Å
            d2   = C[2:] - C[:-2]; d2n = np.linalg.norm(d2, axis=1) + 1e-6
            adj2 = d2 * ((10.2 - d2n) / d2n)[:, None] * (0.10 * strength)
            C[:-2] -= adj2; C[2:] += adj2
            # Laplacian smoothing
            C[1:-1] += (0.06 * strength) * (0.5 * (C[:-2] + C[2:]) - C[1:-1])
            # self-avoidance
            if L >= 25:
                idx  = np.linspace(0, L - 1, min(L, 160)).astype(int) if L > 220 else np.arange(L)
                P    = C[idx]; diff = P[:, None, :] - P[None, :, :]
                dm   = np.linalg.norm(diff, axis=2) + 1e-6
                sep  = np.abs(idx[:, None] - idx[None, :])
                mask = (sep > 2) & (dm < 3.2)
                if np.any(mask):
                    vec = (diff * ((3.2 - dm) / dm)[:, :, None] * mask[:, :, None]).sum(axis=1)
                    C[idx] += (0.015 * strength) * vec
            X[s:e] = C
    return X


def _rotmat(axis, ang):
    a = np.asarray(axis, float); a /= np.linalg.norm(a) + 1e-12
    x, y, z = a; c, s = np.cos(ang), np.sin(ang); CC = 1 - c
    return np.array([[c+x*x*CC, x*y*CC-z*s, x*z*CC+y*s],
                     [y*x*CC+z*s, c+y*y*CC, y*z*CC-x*s],
                     [z*x*CC-y*s, z*y*CC+x*s, c+z*z*CC]])


def apply_hinge(coords, seg, rng, deg=22):
    s, e = seg; L = e - s
    if L < 30: return coords
    pivot = s + int(rng.integers(10, L - 10))
    R = _rotmat(rng.normal(size=3), np.deg2rad(float(rng.uniform(-deg, deg))))
    X = coords.copy(); p0 = X[pivot].copy()
    X[pivot+1:e] = (X[pivot+1:e] - p0) @ R.T + p0
    return X


def jitter_chains(coords, segs, rng, deg=12, trans=1.5):
    X = coords.copy(); gc_ = X.mean(0, keepdims=True)
    for s, e in segs:
        R     = _rotmat(rng.normal(size=3), np.deg2rad(float(rng.uniform(-deg, deg))))
        shift = rng.normal(size=3); shift = shift / (np.linalg.norm(shift) + 1e-12) * float(rng.uniform(0, trans))
        c     = X[s:e].mean(0, keepdims=True)
        X[s:e] = (X[s:e] - c) @ R.T + c + shift
    X -= X.mean(0, keepdims=True) - gc_
    return X


def smooth_wiggle(coords, segs, rng, amp=0.8):
    X = coords.copy()
    for s, e in segs:
        L = e - s
        if L < 20: continue
        ctrl = np.linspace(0, L - 1, 6); disp = rng.normal(0, amp, (6, 3)); t = np.arange(L)
        X[s:e] += np.vstack([np.interp(t, ctrl, disp[:, k]) for k in range(3)]).T
    return X

def generate_rna_structure(sequence: str, seed=None) -> np.ndarray:
    """Bounded super-helix initialization — prevents rod explosion on megascale targets.

    Design:
      * Inner helix : radius 10 Å, angular step 0.6 rad/residue (A-form pitch).
      * Super-helix : winds the strand into a slowly-rotating torus so that
        x/y stay within ±55 Å of the origin regardless of sequence length.
      * Z-axis wraps every 400 residues (z_max ≈ 598 Å per period), so even a
        4,640-nt RNA never exceeds z ~ ±306 Å after centering — well inside the
        ±9,999.999 CSV clip.

    Old version: z = i * 2.5  →  z_max = 11,600 Å at N=4,640  →  CLIPS.
    """
    if seed is not None:
        np.random.seed(seed)
    n = len(sequence)
    coords = np.zeros((n, 3))
    for i in range(n):
        ang       = i * 0.6
        super_ang = i * 0.05
        super_r   = 30.0 + 15.0 * np.sin(i * 0.01)
        coords[i] = [
            super_r * np.cos(super_ang) + 10.0 * np.cos(ang),
            super_r * np.sin(super_ang) + 10.0 * np.sin(ang),
            (i % 400) * 1.5,   # wraps z: max ~598 Å/period
        ]
    return coords

# ─────────────── Adaptive TBM Identity Threshold ─────────────────────────────
def get_min_pct_identity(seq_len: int) -> float:
    """
    Dynamic identity threshold based on sequence length.
    Short RNAs benefit from a lower threshold — more templates qualify.
    """
    if seq_len < 80:
        return 45.0
    else:
        return 55.0


# ─────────────── Diverse Structure Picker ────────────────────────────────────
def pick_diverse(coords: np.ndarray, k: int) -> np.ndarray:
    """
    Greedy max-min diversity selection from a pool of predicted structures.

    Always called as pick_diverse(coords, min(N_SAMPLE, coords.shape[0])) so
    the entire over-generated pool (e.g. 6 candidates) is considered before
    trimming to the final N_SAMPLE slots.  Using n_needed here instead would
    throw away most of the diversity benefit.
    """
    if coords.shape[0] <= k:
        return coords

    chosen    = [coords[0]]
    remaining = list(range(1, coords.shape[0]))

    while len(chosen) < k and remaining:
        dists = []
        for idx in remaining:
            c        = coords[idx]
            min_dist = min(np.mean(np.linalg.norm(c - x, axis=1)) for x in chosen)
            dists.append(min_dist)

        best_idx = remaining[int(np.argmax(dists))]
        chosen.append(coords[best_idx])
        remaining.remove(best_idx)

    return np.stack(chosen)


# ─────────────── TBM Phase ───────────────────────────────────────────────────
def tbm_phase(test_df, train_seqs_df, train_coords_dict, segments_map):
    """
    Phase 1 — Template-Based Modeling.

    Returns
    -------
    template_predictions : {target_id: [np.ndarray(seq_len, 3), ...]}
        0 to N_SAMPLE predictions per target, from real templates.
    protenix_queue : {target_id: (n_needed, full_sequence)}
        Targets that need Protenix (missing slots, or best_sim < 0.55).
    """
    print(f"\n{'='*60}")
    print(f"PHASE 1: Template-Based Modeling")
    print(f"  MIN_SIMILARITY = {MIN_SIMILARITY}  |  MIN_PCT_IDENTITY = adaptive")
    print(f"{'='*60}")
    t0 = time.time()

    template_predictions: dict = {}
    protenix_queue:       dict = {}

    for _, row in test_df.iterrows():
        tid  = row["target_id"]
        seq  = row["sequence"]
        segs = segments_map.get(tid, [(0, len(seq))])

        # Adaptive identity threshold per sequence length
        min_pct = get_min_pct_identity(len(seq))
        similar = find_similar_sequences_detailed(seq, train_seqs_df, train_coords_dict, top_n=30)
        preds   = []
        used    = set()

        for i, (tmpl_id, tmpl_seq, sim, tmpl_coords, pct_id, _, _) in enumerate(similar):
            if len(preds) >= N_SAMPLE:
                break
            if sim < MIN_SIMILARITY or pct_id < min_pct:
                break   # sorted by sim — no point continuing
            if tmpl_id in used:
                continue

            rng     = np.random.default_rng((row.name * 10000000000 + i * 10007) % (2**32))
            adapted = adapt_template_to_query(seq, tmpl_seq, tmpl_coords)
            slot    = len(preds)

            # ── Diversity transforms — correct slot order ─────────────────
            # slot 0 → pure template (safest, lowest energy state)
            # slot 1 → noise             (early noise improves spread metric)
            # slot 2 → hinge             (large structural perturbation)
            # slot 3 → jitter chains     (rigid-body diversity)
            # slot 4 → smooth wiggle     (continuous deformation)
            
            if slot == 0:
                X = adapted
            elif slot == 1:
                X = adapted + rng.normal(0, max(0.01, (0.40 - sim) * 0.06), adapted.shape)
            elif slot == 2:
                longest = max(segs, key=lambda se: se[1] - se[0])
                X = apply_hinge(adapted, longest, rng)
            elif slot == 3:
                X = jitter_chains(adapted, segs, rng)
            else:
                X = smooth_wiggle(adapted, segs, rng)

            refined = adaptive_rna_constraints(X, tid, segments_map, confidence=sim)
            preds.append(refined)
            used.add(tmpl_id)

        template_predictions[tid] = preds
        n_needed = N_SAMPLE - len(preds)
        best_sim = similar[0][2] if similar else 0.0

        # FIX: threshold 0.55 (was 0.65) — avoid over-routing good TBM targets.
        # Only supplement with Protenix when TBM is genuinely weak.
        if n_needed > 0 or (len(preds) > 0 and best_sim < 0.55):
            protenix_queue[tid] = (max(2, n_needed), seq)
            print(f"  {tid} ({len(seq)} nt, min_pct={min_pct}): "
                  f"{len(preds)} TBM (best_sim={best_sim:.3f}) "
                  f"→ queuing {max(2, n_needed)} Protenix samples")
        else:
            print(f"  {tid} ({len(seq)} nt, min_pct={min_pct}): "
                  f"all {N_SAMPLE} from TBM ✓ (best_sim={best_sim:.3f})")

    elapsed = time.time() - t0
    n_full  = len(test_df) - len(protenix_queue)
    print(f"\nPhase 1 done in {elapsed:.1f}s")
    print(f"  Fully covered by TBM : {n_full}")
    print(f"  Routed to Protenix   : {len(protenix_queue)}")
    return template_predictions, protenix_queue


# ─────────────── Main ────────────────────────────────────────────────────────

print('TBM functions loaded')


TBM functions loaded


In [20]:
import sys, os, importlib, glob

ROOT_PATH = PROTENIX_CODE_DIR

if not os.path.isdir(ROOT_PATH):
    print("⚠️ Primary Protenix path not found. Searching...")
    _search = glob.glob("/kaggle/input/**/Protenix-v1", recursive=True)
    if _search:
        ROOT_PATH = _search[0]
        print(f"✅ Found: {ROOT_PATH}")
    else:
        print("❌ CRITICAL: Could not find Protenix-v1 folder.")

if ROOT_PATH not in sys.path:
    sys.path.insert(0, ROOT_PATH)

# Change directory so Protenix can find its relative local files
os.chdir(ROOT_PATH)
print(f"🏠 Working directory: {os.getcwd()}")

# ── REMOVED the read-only file writing loop here. 
# ── The previous cell already verified the packages!

try:
    import configs
    from protenix.data.inference.infer_dataloader import InferenceDataset
    from runner.inference import InferenceRunner
    print("🚀 ALL SYSTEMS GO! Protenix modules linked successfully in working dir.")
except ImportError as e:
    print(f"❌ Standard import failed: {e}")
    if "configs" in str(e):
        try:
            _spec = importlib.util.spec_from_file_location(
                "configs", os.path.join(ROOT_PATH, "configs", "__init__.py"))
            _mod = importlib.util.module_from_spec(_spec)
            sys.modules["configs"] = _mod
            _spec.loader.exec_module(_mod)
            print("✅ Configs module manually injected.")
        except Exception as _fe:
            print(f"❌ Forced injection failed: {_fe}")

🏠 Working directory: /kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1
🚀 ALL SYSTEMS GO! Protenix modules linked successfully in working dir.


In [21]:
# ============================================================
# SHR PHYSICS ENGINE (JAX ACCELERATED) - 64-BIT PRECISION
# Stochastic Hamiltonian Relaxation (Manuscript V8.0)
#
# Operates on C1' coordinates: (N, 3) per structure
# Hamiltonian: H(x) = E_bond + E_rep + E_DL + E_Rg
# Sobolev H1 gradient preconditioning via JAX DCT-II
# ============================================================
import jax
import jax.numpy as jnp
import jax.scipy.fft as jfft
import numpy as np
import time

# 1. Enable 64-bit precision for physical accuracy
jax.config.update("jax_enable_x64", True)

# --- Constants ---
D0_BOND = 5.95
K_BOND = 10.0
SIGMA_CLASH = 3.0
D_CONTACT = 8.0

# --- Dynamic Parameters ---
def get_shr_params(N):
    """
    Dynamic parameter schedule based on sequence length L = N nucleotides.

    kdl:         Linear 10.0 (N=1000) -> 25.0 (N=4640).
                 Compaction gravity needed to reach Rg ~104 A for 9MME.
    sigma_clash: Linear 3.0 A (N=1000) -> 2.6 A (N=4640).
                 Tighter packing without steric blow-up.
    k_bond:      Fixed 100.0 for all tiers — prevents chain breakage at
                 high-force compaction levels.
    """
    # Fraction along the 1000->4640 nt axis, clamped to [0, 1]
    t = min(max((N - 1000) / (4640 - 1000), 0.0), 1.0)

    if N > 3000:        # ── Megascale tier (e.g. 9MME at 4640 nt) ───────────
        kdl         = 10.0 + 15.0 * t       # 10.0 -> 25.0
        sigma_clash = 3.0  -  0.4 * t       # 3.0  ->  2.6
        return dict(
            K=1,
            T=8000,
            lr0=5e-3,
            w_DL=round(kdl, 3),
            clip=2.0,
            sigma_J=15.0,
            k_bond=100.0,
            sigma_clash=round(sigma_clash, 3),
        )
    elif N > 1000:      # ── Large tier ───────────────────────────────────────
        kdl         = 10.0 + 15.0 * t
        sigma_clash = 3.0  -  0.4 * t
        return dict(
            K=3,
            T=1500,
            lr0=1e-2,
            w_DL=round(kdl, 3),
            clip=5.0,
            sigma_J=10.0,
            k_bond=100.0,
            sigma_clash=round(sigma_clash, 3),
        )
    else:               # ── Standard tier (N <= 1000) ────────────────────────
        return dict(
            K=5,
            T=1000,
            lr0=2e-2,
            w_DL=10.0,
            clip=5.0,
            sigma_J=5.0,
            k_bond=100.0,
            sigma_clash=3.0,
        )


# --- Energy Functions (JAX) ---
def energy_bond(coords, d0=5.95, k_bond=10.0):
    diffs = coords[1:] - coords[:-1]
    dists = jnp.sqrt(jnp.sum(diffs**2, axis=-1) + 1e-8)
    return k_bond * jnp.sum((dists - d0)**2)

def energy_steric(coords, sigma_clash=3.0):
    diff = coords[:, None, :] - coords[None, :, :]
    # Epsilon raised from 1e-8 → 1e-2 (minimum effective distance = 0.1 Å).
    # The steric gradient at ||diff|| → 0 scales as 2*sigma_clash / sqrt(eps):
    #   eps=1e-8  →  max grad ≈ 60,000 per pair/coord  (causes explosion)
    #   eps=1e-2  →  max grad ≈     60 per pair/coord  (1000× safer)
    # Physical distances are never below ~1 Å in real structures, so raising
    # eps to 1e-2 leaves the energy landscape identical for all valid configs.
    dist = jnp.sqrt(jnp.sum(diff**2, axis=-1) + 1e-2)
    n = coords.shape[0]
    i_idx = jnp.arange(n)
    mask = (i_idx[None, :] > i_idx[:, None] + 1)
    violations = jnp.maximum(sigma_clash - dist, 0.0)
    return jnp.sum((violations * mask)**2)

def energy_contacts(coords, contact_map, d_contact=8.0, w_DL=10.0):
    diff = coords[:, None, :] - coords[None, :, :]
    dists = jnp.sqrt(jnp.sum(diff**2, axis=-1) + 1e-8)
    violations = jnp.maximum(dists - d_contact, 0.0)
    energy = jnp.sum(contact_map * (violations**2))
    return w_DL * energy

def energy_rg(coords, rg_target, k_rg=1.0):
    center = jnp.mean(coords, axis=0)
    rg_sq = jnp.mean(jnp.sum((coords - center)**2, axis=-1))
    rg = jnp.sqrt(rg_sq + 1e-8)
    violation = jnp.maximum(rg_target - rg, 0.0)
    return k_rg * (violation**2)

def total_energy(coords, contact_map, params):
    # Pull dynamic force constants from params dict.
    # Defaults maintain backward-compatibility with callers that pre-date v14.
    k_bond      = params.get('k_bond',      100.0)
    sigma_clash = params.get('sigma_clash',   3.0)
    e_b = energy_bond(coords, d0=5.95, k_bond=k_bond)
    e_s = energy_steric(coords, sigma_clash=sigma_clash)
    e_c = energy_contacts(coords, contact_map, d_contact=8.0, w_DL=params['w_DL'])
    e_r = energy_rg(coords, params['rg_target'], k_rg=1.0)
    return e_b + e_s + e_c + e_r

# --- Sobolev Smoothing (JAX FFT) ---
def sobolev_h1_smooth(gradient, alpha=10.0):
    N = gradient.shape[0]
    k = jnp.arange(N)
    inv_sigma = 1.0 / (1.0 + alpha * k**2)
    inv_sigma = inv_sigma[:, None]
    g_hat = jfft.dct(gradient, type=2, norm='ortho', axis=0)
    g_hat_filtered = g_hat * inv_sigma
    smoothed_gradient = jfft.idct(g_hat_filtered, type=2, norm='ortho', axis=0)
    return smoothed_gradient

# --- Update Step (JIT) ---
@jax.jit
def step_fn(coords, params):
    grads = jax.grad(total_energy)(coords, params['contact_map'], params)
    smooth_grads = sobolev_h1_smooth(grads, alpha=params['alpha'])
    clipped_grads = jnp.clip(smooth_grads, -params['clip'], params['clip'])
    new_coords = coords - params['lr'] * clipped_grads
    return new_coords

# --- Refinement Trajectories ---
def shr_refine_single(coords_init, contact_map=None, params=None, seed=0, verbose=False):
    if params is None:
        N = len(coords_init)
        params = get_shr_params(N)

    coords_jax = jnp.array(coords_init, dtype=jnp.float64)
    if contact_map is None:
        contact_map = jnp.zeros((len(coords_init), len(coords_init)), dtype=jnp.float64)
    else:
        contact_map = jnp.array(contact_map, dtype=jnp.float64)

    N = len(coords_init)
    rg_target = 3.5 * (N ** 0.45) if N > 200 else 0.0

    sim_params = {
        'lr':          params['lr0'],
        'alpha':       10.0,
        'clip':        params.get('clip',        5.0),
        'w_DL':        params.get('w_DL',        5.0),
        'k_bond':      params.get('k_bond',    100.0),
        'sigma_clash': params.get('sigma_clash',  3.0),
        'rg_target':   rg_target,
        'contact_map': contact_map,
    }

    key = jax.random.PRNGKey(seed)
    jitter = jax.random.normal(key, coords_jax.shape) * params.get('sigma_J', 10.0)
    current_coords = coords_jax + jitter
    current_coords -= jnp.mean(current_coords, axis=0)

    T = params['T']
    lr0 = params['lr0']

    def scan_body(carrier, t):
        coords = carrier
        effective_lr = lr0 * (1.0 - t / T)
        step_params = sim_params.copy()
        step_params['lr'] = effective_lr
        new_coords = step_fn(coords, step_params)
        return new_coords, None

    final_coords, _ = jax.lax.scan(scan_body, current_coords, jnp.arange(T))
    final_energy = total_energy(final_coords, contact_map, sim_params)
    return np.array(final_coords), float(final_energy)

def shr_polish(coords, contact_map=None, n_steps=2000, lr=0.01, verbose=False):
    coords_jax = jnp.array(coords, dtype=jnp.float64)
    if contact_map is None:
        contact_map = jnp.zeros((len(coords), len(coords)), dtype=jnp.float64)
    else:
        contact_map = jnp.array(contact_map, dtype=jnp.float64)

    polish_params = {
        'lr': lr,
        'alpha': 5.0,
        'clip': 2.0,
        'w_DL': 2.0,
        'rg_target': 0.0,
        'contact_map': contact_map
    }

    def scan_body(carrier, _):
        return step_fn(carrier, polish_params), None

    final_coords, _ = jax.lax.scan(scan_body, coords_jax, None, length=n_steps)
    return np.array(final_coords)

print("SHR Physics Engine upgraded to JAX (x64 precision enabled).")

SHR Physics Engine upgraded to JAX (x64 precision enabled).


In [22]:
import os

_CHECKS = [
    ("RNA-FM weights",        RNA_FM_WEIGHTS),
    ("RibonanzaNet-2 weights",RIBONANZANET2_WEIGHTS),
    ("RibonanzaNet-2 yaml",   RIBONANZANET2_YAML),
    ("Boltz-1 checkpoint",    BOLTZ_CHECKPOINT),
    ("Boltz-1 CCD",           BOLTZ_CCD),
    ("Protenix checkpoint",   PROTENIX_CHECKPOINT),
]
print("── Model Weight Check ──")
for _label, _path in _CHECKS:
    if os.path.isfile(_path):
        _sz = os.path.getsize(_path) / 1024**2
        print(f"  ✅ {_label}: {_sz:.0f} MB")
    else:
        print(f"  ❌ {_label}: NOT FOUND — {_path}")

print("\n── Boltz Cache Check ──")
for _f in ["boltz1_conf.ckpt", "ccd.pkl"]:
    _p = os.path.join(BOLTZ_CACHE_DIR, _f)
    print(f"  {'✅' if os.path.isfile(_p) else '❌'} {_f}")


── Model Weight Check ──
  ✅ RNA-FM weights: 1139 MB
  ✅ RibonanzaNet-2 weights: 387 MB
  ✅ RibonanzaNet-2 yaml: 0 MB
  ✅ Boltz-1 checkpoint: 3429 MB
  ✅ Boltz-1 CCD: 330 MB
  ✅ Protenix checkpoint: 1408 MB

── Boltz Cache Check ──
  ✅ boltz1_conf.ckpt
  ✅ ccd.pkl


In [23]:
# ============================================================
# ENHANCED MAIN PIPELINE
# Phase 0: Pre-load contact models (RNA-FM + RibonanzaNet-2)
# Phase 1: Adaptive TBM
# Phase 2a: Protenix + Boltz-1 Ensemble (≤512 nt, GPU 1, sequential)
# Phase 2b: SHR Megascale (>512 nt, GPU 0)
# Phase 3: SHR Physics Polish (blended contact maps)
# Phase 4: Combine TBM + Boltz-1 + Protenix + Megascale + de-novo
# ============================================================

import pandas as pd

SENTINEL = -2_000_000.0  # v19: global sentinel constant

# ── SHR Configuration ─────────────────────────────────────────────────────────
SHR_ENABLED       = True
SHR_POLISH_TBM    = True
SHR_POLISH_PTX    = True
SHR_POLISH_STEPS  = 8000
SHR_FULL_SEEDS    = 5
MEGASCALE_THRESH  = 512
BOLTZ_MAX_LEN     = 800     # OOM guard for T4

# Contact map cache (stores blended maps)
_contact_map_cache = {}


# ── Blended contact map builder ───────────────────────────────────────────────
def get_or_build_contact_map(sequence, target_id, train_coords_dict=None,
                              train_seqs_df=None, segments_map=None):
    """
    Build a consensus contact map for `sequence` using a three-tier strategy:

    Tier 1 (preferred):
      • Generate RNA-FM HWS contact map  (cosine similarity → threshold)
      • Generate RibonanzaNet-2 pairwise contact map (sigmoid of mean features)
      • Blend:  0.5 * rna_fm_cmap + 0.5 * ribonanza_cmap
      The blended map is richer than either alone and feeds JAX E_DL.

    Tier 2 (fallback — if RNA-FM unavailable):
      • Template-derived contacts from nearest training structure.

    Tier 3 (last resort):
      • Backbone-only (consecutive C1'-C1' contacts).

    Result is cached per target_id.
    """
    if target_id in _contact_map_cache:
        return _contact_map_cache[target_id]

    N    = len(sequence)
    cmap = None

    # ── Tier 1: RNA-FM HWS ───────────────────────────────────────────────────
    rna_fm_cmap = get_contact_map_for_sequence(sequence, threshold=0.85)

    if rna_fm_cmap is not None:
        print(f"  {target_id}: RNA-FM HWS contact map ({N}×{N}) ✅")

        # ── Also run RibonanzaNet-2 ───────────────────────────────────────────
        # ── RibonanzaNet-2: HWS-safe inference ───────────────────────────────
        # For N <= 1022: delegates internally to the single-pass function.
        # For N > 1022: sliding-window accumulator -- a 4,640-nt sequence that
        #   would need 82 GB in one pass becomes ~30 sequential ~1.5 GB windows.
        ribonanza_cmap = get_ribonanzanet_contacts_hws(
            sequence,
            model=_RIBONANZA_MODEL,
            device=DEVICE_RIBONANZA,
            window_size=RN2_HWS_WINDOW,   # 1022 nt
            stride=RN2_HWS_STRIDE,        # 768 nt -> 254-nt overlap
        )

        if ribonanza_cmap is not None:
            # 50/50 blend: both maps are float32 numpy in [0, 1].
            # RNA-FM: co-evolutionary embedding cosine similarity.
            # RibonanzaNet-2: learned pairwise structural feature predictions.
            # Their union gives richer E_DL restraints than either alone.
            cmap = 0.5 * rna_fm_cmap + 0.5 * ribonanza_cmap
            print(f"  {target_id}: blended (RNA-FM 50% + RibonanzaNet-2 HWS 50%)")
        else:
            cmap = rna_fm_cmap
            print(f"  {target_id}: using RNA-FM only (RibonanzaNet-2 HWS unavailable)")

        _contact_map_cache[target_id] = cmap
        return cmap

    # ── Tier 2: Template-derived contacts ────────────────────────────────────
    if train_coords_dict is not None and train_seqs_df is not None:
        try:
            similar = find_similar_sequences_detailed(
                sequence, train_seqs_df, train_coords_dict, top_n=3
            )
            if similar:
                adapted = adapt_template_to_query(sequence, similar[0][1], similar[0][3])
                cmap = extract_contacts_from_template(adapted, d_cutoff=12.0, min_sep=6)
                if cmap is not None and np.sum(cmap) > 0:
                    print(f"  {target_id}: template contacts "
                          f"({int(np.sum(np.triu(cmap, k=1)))} pairs)")
                    _contact_map_cache[target_id] = cmap
                    return cmap
        except Exception:
            pass

    # ── Tier 3: Backbone fallback ────────────────────────────────────────────
    cmap = np.zeros((N, N), dtype=np.float32)
    for i in range(N - 1):
        cmap[i, i+1] = cmap[i+1, i] = 1.0
    _contact_map_cache[target_id] = cmap
    return cmap


def load_cached_diversity_models(target_id, cache_dir="/kaggle/input/my-precomputed-rna", n_samples=5):
    """Dormant: check for pre-computed .pdb files for a massive target."""
    from Bio import PDB
    parser = PDB.PDBParser(QUIET=True)
    for i in range(1, n_samples + 1):
        if not os.path.exists(f"{cache_dir}/{target_id}_model_{i}.pdb"):
            return None
    print(f"✅ Cache HIT for {target_id}!")
    cached_coords = []
    for i in range(1, n_samples + 1):
        structure = parser.get_structure(target_id, f"{cache_dir}/{target_id}_model_{i}.pdb")
        coords = []
        for model in structure:
            for chain in model:
                for residue in chain:
                    if "C1'" in residue:
                        coords.append(residue["C1'"].get_coord())
        cached_coords.append(np.array(coords))
    return np.array(cached_coords)



# ── Homomeric structure utilities ──────────────────────────────────────────
# FIX 2.4: These were previously undefined, causing silent NameError on every
# megascale target and skipping the full block-diagonal contact map build.

def detect_homomeric_structure(sequence: str, segments: list) -> "int | None":
    """
    Detect whether a multi-segment RNA is a homomeric assembly.

    A structure is considered homomeric when:
      - There are 2 or more segments.
      - All segments have the same length (within ±2 nt to allow for
        minor trimming artefacts in the segment builder).
      - The subsequences for each segment are identical at the nucleotide
        level (>= 95% identity via Smith-Waterman score shortcut).

    Returns
    -------
    monomer_len : int   if homomeric  (length of one chain)
    None               if not homomeric or only one segment
    """
    if segments is None or len(segments) < 2:
        return None

    seg_lengths = [e - s for s, e in segments]
    ref_len = seg_lengths[0]

    # All segments must be the same length (±2 nt tolerance)
    if not all(abs(l - ref_len) <= 2 for l in seg_lengths):
        return None

    # Sequence identity check: compare each chain to the first
    ref_seq = sequence[segments[0][0]: segments[0][1]]
    for s, e in segments[1:]:
        chain_seq = sequence[s:e]
        # Align over the shorter of the two
        min_len = min(len(ref_seq), len(chain_seq))
        if min_len == 0:
            return None
        matches = sum(a == b for a, b in zip(ref_seq[:min_len], chain_seq[:min_len]))
        identity = matches / min_len
        if identity < 0.95:
            return None  # Not identical enough — heteromeric or chimeric

    print(f"  [Homomeric] Detected {len(segments)}-chain homomeric assembly "
          f"(monomer={ref_len} nt, identity≥95%)")
    return ref_len


def build_block_diagonal_contacts(
    mono_cmap: "np.ndarray",
    n_chains: int,
    monomer_len: int,
) -> "np.ndarray":
    """
    Tile a monomer contact map into a block-diagonal full-complex map.

    For an N-chain homomeric assembly the contact map is:
      [ mono  0    0   ... ]
      [ 0    mono  0   ... ]
      [ 0    0    mono ... ]

    This correctly represents intra-chain contacts while setting all
    inter-chain contacts to zero (conservative — we do not model
    inter-chain packing restraints for symmetric assemblies).

    Parameters
    ----------
    mono_cmap   : (monomer_len, monomer_len) float32 contact map
    n_chains    : number of identical chains
    monomer_len : length of each chain

    Returns
    -------
    full_cmap : (n_chains * monomer_len, n_chains * monomer_len) float32
    """
    full_len  = n_chains * monomer_len
    full_cmap = np.zeros((full_len, full_len), dtype=np.float32)
    for k in range(n_chains):
        s = k * monomer_len
        e = s + monomer_len
        # Ensure mono_cmap fits exactly (trim if segment length had ±2 nt slack)
        actual = min(monomer_len, mono_cmap.shape[0])
        full_cmap[s:s+actual, s:s+actual] = mono_cmap[:actual, :actual]
    print(f"  [Homomeric] Block-diagonal contact map: "
          f"{full_len}×{full_len} ({n_chains} × {monomer_len} nt)")
    return full_cmap

def handle_megascale_target(target_id, sequence, segments_map, train_coords_dict,
                             train_seqs_df):
    """Handle targets >512 nt via full SHR with blended contact maps.

    v17 changelog (Double Trap fix):
      1. UNCONDITIONAL Seed-0 Crystal Guard — protects slot 0 regardless of
         whether init_coords came from TBM or HWS stitching.
      2. Per-seed KDTree Armor inside the diversity loop — prevents JAX OOM
         from clashing atoms in stitched megascale models.
      3. Per-seed JAX RAM clear — flushes backends + gc between seeds so a
         corrupted grad graph from seed k cannot cascade into seed k+1.
    """
    N = len(sequence)
    segments = segments_map.get(target_id, [(0, N)])
    print(f"\n  MEGASCALE: {target_id} ({N} nt, {len(segments)} segment(s))")

    cmap = get_or_build_contact_map(
        sequence, target_id, train_coords_dict, train_seqs_df, segments_map
    )

    try:
        monomer_len = detect_homomeric_structure(sequence, segments)
        if monomer_len is not None:
            n_chains = len(segments)
            print(f"  Homomeric: {n_chains} chains × {monomer_len} nt")
            mono_seq  = sequence[:monomer_len]
            mono_cmap = get_contact_map_for_sequence(mono_seq, threshold=0.85)
            if mono_cmap is None:
                mono_cmap = np.zeros((monomer_len, monomer_len), dtype=np.float32)
                for i in range(monomer_len - 1):
                    mono_cmap[i, i+1] = mono_cmap[i+1, i] = 1.0
            cmap = build_block_diagonal_contacts(mono_cmap, n_chains, monomer_len)
    except NameError:
        pass

    # --- 3D OVERLAP STITCHING INITIALIZATION ---
    import uuid
    import shutil  # already imported in Cell 16, harmless to re-import

    def boltz_predictor_wrapper(sub_seq: str) -> np.ndarray:
        chunk_id = f"chunk_{uuid.uuid4().hex[:8]}"
        temp_dir = f"/kaggle/working/stitch_temp_{target_id}"
        os.makedirs(temp_dir, exist_ok=True)
        coords = generate_boltz_seed(
            target_id=chunk_id,
            sequence=sub_seq,
            out_dir=temp_dir,
            device=DEVICE_BOLTZ,
        )
        if coords is None or len(coords) != len(sub_seq):
            raise RuntimeError("Boltz failed to generate valid coordinates for chunk.")
        return coords

    # ── 1. Try Template Matching First ────────────────────────────────────────
    try:
        similar = find_similar_sequences_detailed(
            sequence, train_seqs_df, train_coords_dict, top_n=1
        )
        # Require >= 50% normalised similarity to trust a megascale template
        if similar and similar[0][2] > 0.50:
            init_coords = adapt_template_to_query(sequence, similar[0][1], similar[0][3])
            print(f"  Init: High-confidence template found (sim={similar[0][2]:.3f})")
            stitching_needed = False

            # ── TEMPLATE TRUST: suppress ML contact map for near-perfect templates ──
            if similar[0][2] > 0.98:
                # cmap = cmap * 0.1  <-- REMOVED v15: was braking compaction

                # FIX 1.2/1.3: Overwrite the global cache with the scaled-down map so
                # that Phase 3 shr_polish re-fetches the 10% map, not the full-strength
                # one that previously catapulted 9MME atoms 176,000 Å away.
                # The previous code used 'contact_map_cache' (missing underscore), which
                # raised a silent NameError absorbed by the outer except Exception block,
                # discarding init_coords and routing to the Boltz stitcher instead.
                _contact_map_cache[target_id] = cmap

                print(
                    f"  ⚠️  Template Trust active (sim={similar[0][2]:.3f} > 0.98): "
                    f"contact map scaled ×0.1 — cache updated to protect crystal template."
                )
        else:
            stitching_needed = True
    except Exception:
        stitching_needed = True

    # ── 2. Fallback to 3D Overlap Stitching ──────────────────────────────────
    if stitching_needed:
        print(f"  Init: 3D Overlap Stitching via Boltz-1...")
        try:
            init_coords = stitch_megascale_rna(
                sequence=sequence,
                predictor_fn=boltz_predictor_wrapper,
                window_size=512,
                overlap=128,
                verbose=False,
            )
            print(f"  Init: Stitching complete ✅ ({init_coords.shape[0]} atoms)")
            shutil.rmtree(f"/kaggle/working/stitch_temp_{target_id}", ignore_errors=True)
        except Exception as _se:
            print(f"  Init: Stitching failed ({_se}) — falling back to A-form helix")
            init_coords = generate_rna_structure(sequence, seed=42)
    # ─────────────────────────────────────────────────────────────────────────
    # -------------------------------------------

    # ── Stage-1: universal micro-jitter (σ=0.1 Å, ALL residues) ────────────────
    # Breaks exact coincidences AND near-degeneracies that an == 0.0 mask
    # misses.  0.1 Å RMS << bond-deviation scale (±0.5 Å) so a high-quality
    # template is not meaningfully perturbed.
    _rng_init = np.random.default_rng(42)
    init_coords = init_coords.copy()
    init_coords += _rng_init.normal(0.0, 0.1, init_coords.shape).astype(np.float32)

    # ── Stage-2: targeted clash-jitter via KDTree (O(N log N)) ────────────────
    # Detect non-sequential residue pairs within sigma_clash/2 = 1.5 Å.
    # These generate the largest steric gradients and can drive JAX scan
    # explosion over 8,000 steps for a 4,640-nt RNA with ~10.7M steric pairs.
    try:
        from scipy.spatial import KDTree as _KDTree
        _CLASH_R = 1.5   # Å  (= sigma_clash / 2)
        _close   = _KDTree(init_coords).query_pairs(_CLASH_R)
        _cset    = set()
        for _pi, _pj in _close:
            if abs(_pi - _pj) > 1:   # skip sequential backbone neighbours
                _cset.add(_pi); _cset.add(_pj)
        if _cset:
            _cidx = np.array(sorted(_cset), dtype=int)
            init_coords[_cidx] = _rng_init.normal(
                0.0, 3.0, (len(_cidx), 3)
            ).astype(np.float32)
            print(f"  Init: clash-jitter → {len(_cidx)} residues "
                  f"({len(_close)} pairs within {_CLASH_R} Å)")
        else:
            print(f"  Init: no non-sequential clashes < {_CLASH_R} Å "
                  f"(stage-2 jitter not needed)")
    except Exception as _kje:
        # zero-mask fallback (original behaviour)
        print(f"  Init: KDTree unavailable ({_kje}); using zero-mask fallback")
        _zero_mask = np.all(init_coords == 0.0, axis=1)
        if _zero_mask.any():
            init_coords[_zero_mask] = _rng_init.normal(
                0.0, 2.0, (_zero_mask.sum(), 3)
            ).astype(np.float32)
            print(f"  Init: zero-jitter → {_zero_mask.sum()} residues")

    N_DIVERSE = 5; BASE_SEED = 42; results = []; progress_log = []
    params = get_shr_params(N)
    print(f"  Diversity loop: {N_DIVERSE} seeds × T={params['T']} steps")

    try:
        for i in range(N_DIVERSE):
            # ══════════════════════════════════════════════════════════════════
            # v17 FIX 1: UNCONDITIONAL Seed-0 Crystal Guard
            # ══════════════════════════════════════════════════════════════════
            # v16 bug: guard was `if i == 0 and not stitching_needed: continue`
            # For 9MME (0 TBM templates → stitching_needed=True), the guard
            # deactivated.  Seed 0 entered the 25.0-force JAX engine and
            # exploded, corrupting JAX state and cascading OOM into seeds 1-4.
            #
            # v17: Protect Seed 0 ALWAYS.  Even an HWS-stitched model is our
            # best zero-explosion baseline and must be preserved untouched.
            if i == 0:
                print(f"    Seed 0: Preserving coords unconditionally (0 explosions).")
                results.append(init_coords.copy())
                continue

            seed_i = BASE_SEED + i * 7919

            # ── Per-seed coordinate preparation ──────────────────────────────
            if i == 1:
                # First diversity seed gets slight global jitter to explore
                # a nearby basin without drifting far from the init geometry.
                X = init_coords + np.random.default_rng(seed_i).normal(
                    0, 0.024, init_coords.shape
                )
            else:
                X = init_coords.copy()

            # ══════════════════════════════════════════════════════════════════
            # v17 FIX 2: PHASE 2b KDTree ARMOR
            # ══════════════════════════════════════════════════════════════════
            # Apply the same steric armor used in the Stage-2 init block to
            # prevent JAX from crashing/OOMing on clashing atoms in stitched
            # megascale models.  Each seed gets a fresh de-clash pass because
            # the jitter in seed 1 can re-introduce close contacts.
            from scipy.spatial import KDTree
            _close = KDTree(X).query_pairs(1.5)
            _cset = {idx for p1, p2 in _close for idx in (p1, p2) if abs(p1 - p2) > 1}
            if _cset:
                _mask = np.zeros(len(X), dtype=bool)
                _mask[list(_cset)] = True
                X[_mask] += np.random.default_rng(seed_i).normal(
                    0.0, 3.0, (_mask.sum(), 3)
                ).astype(np.float32)
                print(f"    Seed {i}: KDTree armor → {_mask.sum()} clashing residues de-clashed")

            # ══════════════════════════════════════════════════════════════════
            # v17 FIX 3: CLEAR JAX RAM
            # ══════════════════════════════════════════════════════════════════
            # Flush JAX backends + garbage-collect between seeds.  Without
            # this, a corrupted gradient graph from seed k bleeds into seed
            # k+1, triggering the `except Exception` fallback that silently
            # returns uncompacted 132 Å initial coordinates.
            import jax, gc
            jax.clear_backends()
            gc.collect()

            t0 = time.time()
            try:
                coords_i, energy_i = shr_refine_single(
                    X, contact_map=cmap, params=params,
                    seed=seed_i, verbose=False
                )
                rg_i = np.sqrt(np.mean(np.sum((coords_i - coords_i.mean(0))**2, axis=-1)))
                print(f"    Seed {i}: E={energy_i:.1f}, Rg={rg_i:.1f}Å, "
                      f"t={time.time()-t0:.1f}s")
                results.append(coords_i)
                progress_log.append({"target": target_id, "seed": seed_i,
                                     "energy": energy_i, "Rg": rg_i})
                pd.DataFrame(progress_log).to_csv(
                    f"/kaggle/working/{target_id}_recovery.csv", index=False)
            except Exception as exc:
                print(f"    Seed {i}: FAILED — {exc}")
    except Exception as e:
        print(f"Simulation interrupted: {e}")
    finally:
        if progress_log:
            pd.DataFrame(progress_log).to_csv(
                f"/kaggle/working/{target_id}_FINAL.csv", index=False)

    try:
        validate_structure(results[0], target_id=target_id)
    except (NameError, IndexError):
        pass

    rng = np.random.default_rng(BASE_SEED)
    while len(results) < N_DIVERSE:
        base = results[0].copy() if results else init_coords.copy()
        base += rng.normal(0, 0.5, base.shape)
        results.append(base)

    return np.stack(results[:N_DIVERSE], axis=0)


def main() -> None:
    global test_df, template_preds, protenix_preds, boltz_phase2a_preds, \
           megascale_preds, combined_seqs, train_coords, segments_map, _RIBONANZA_MODEL

    # ── Paths ─────────────────────────────────────────────────────────────────
    test_csv   = DEFAULT_TEST_CSV
    train_csv  = DEFAULT_TRAIN_CSV
    val_csv    = DEFAULT_VAL_CSV
    train_lbls = DEFAULT_TRAIN_LBLS
    val_lbls   = DEFAULT_VAL_LBLS
    output_csv = DEFAULT_OUTPUT
    code_dir   = PROTENIX_CODE_DIR
    root_dir   = PROTENIX_CODE_DIR

    if not os.path.isdir(code_dir):
        import glob as _glob
        _found = _glob.glob("/kaggle/input/**/Protenix-v1", recursive=True)
        if _found:
            code_dir = root_dir = _found[0]
        else:
            raise FileNotFoundError(f"Missing PROTENIX_CODE_DIR: {code_dir}")

    os.environ["PROTENIX_ROOT_DIR"] = root_dir
    if code_dir not in sys.path:
        sys.path.insert(0, code_dir)

    seed_everything(SEED)

    # ── Phase 0: Pre-load contact models (GPU 0) ─────────────────────────────
    print(f"\n{'='*60}")
    print("PHASE 0: Pre-loading contact models on GPU 0")
    print(f"{'='*60}")

    # RNA-FM (lazy-loaded later per sequence via HWS; touch it now to warm up)
    _rna_fm_model, _rna_fm_alphabet = load_rna_fm_once(device=DEVICE_RNAFM)

    # RibonanzaNet-2 (load once, keep in memory for all targets)
    _RIBONANZA_MODEL = load_ribonanzanet(device=DEVICE_RIBONANZA)

    # ── Load test data ────────────────────────────────────────────────────────
    test_df_full = pd.read_csv(test_csv)
    test_df      = (test_df_full.head(LOCAL_N_SAMPLES) if not IS_KAGGLE
                    else test_df_full).reset_index(drop=True)
    print(f"\nTest targets: {len(test_df)}"
          + (" (LOCAL MODE)" if not IS_KAGGLE else ""))

    seq_by_id = dict(zip(test_df["target_id"], test_df["sequence"]))

    test_df_trunc = test_df.copy()
    test_df_trunc["sequence"] = test_df_trunc["sequence"].str[:MAX_SEQ_LEN]

    # ── Load training data for TBM ────────────────────────────────────────────
    print("\nLoading training data for TBM ...")
    train_seqs   = pd.read_csv(train_csv)
    val_seqs     = pd.read_csv(val_csv)
    train_labels = pd.read_csv(train_lbls, dtype={6: str})
    val_labels   = pd.read_csv(val_lbls)
    combined_seqs   = pd.concat([train_seqs,   val_seqs],   ignore_index=True)
    combined_labels = pd.concat([train_labels, val_labels], ignore_index=True)
    train_coords    = process_labels(combined_labels)
    segments_map, _ = build_segments_map(test_df)
    print(f"Template pool: {len(combined_seqs)} seqs, {len(train_coords)} structures")

    # ── Phase 1: TBM ─────────────────────────────────────────────────────────
    template_preds, protenix_queue = tbm_phase(
        test_df, combined_seqs, train_coords, segments_map
    )

    # ── Route by length ───────────────────────────────────────────────────────
    megascale_targets = {}
    standard_queue    = {}
    for tid, (n_needed, full_seq) in protenix_queue.items():
        if len(full_seq) > MEGASCALE_THRESH:
            megascale_targets[tid] = (n_needed, full_seq)
            print(f"  {tid}: MEGASCALE ({len(full_seq)} nt) → SHR")
        else:
            standard_queue[tid] = (n_needed, full_seq)

    # ── Phase 2a: Protenix + Boltz-1 Ensemble (GPU 1, strictly sequential) ──────────
    # ─────────────────────────────────────────────────────────────────────────
    # Seed quota split per target:
    #   Protenix → math.ceil(n_needed / 2)    e.g. quota 5→3, 4→2, 3→2, 2→1, 1→1
    #   Boltz-1  → math.floor(n_needed / 2)   e.g. quota 5→2, 4→2, 3→1, 2→1, 1→0
    #
    # Execution order enforced for VRAM safety (both models share cuda:1):
    #   1. ALL Protenix inference (cuda:1)
    #   2. torch.cuda.empty_cache() + gc.collect()    ← explicit VRAM sweep
    #   3. ALL Boltz-1 inference (cuda:1)
    #   4. torch.cuda.empty_cache() + gc.collect()    ← second VRAM sweep
    # ─────────────────────────────────────────────────────────────────────────
    protenix_preds      = {}
    boltz_phase2a_preds = {}

    if standard_queue and USE_PROTENIX:
        import math as _math
        print(f"\n{'='*60}")
        print(f"PHASE 2a: Protenix + Boltz-1 Ensemble for "
              f"{len(standard_queue)} standard targets")
        print(f"{'='*60}")

        # ── Compute per-target seed splits ────────────────────────────────────
        # _split[tid] = (n_ptx, n_boltz, n_needed, full_seq)
        _split = {}
        for _tid, (_nn, _fs) in standard_queue.items():
            _np = _math.ceil(_nn / 2)
            _nb = _nn - _np
            _split[_tid] = (_np, _nb, _nn, _fs)
            print(f"  {_tid} ({len(_fs)} nt): quota={_nn} "
                  f"→ {_np} Protenix + {_nb} Boltz-1")

        _boltz_phase2a_dir = "/kaggle/working/boltz_phase2a"
        os.makedirs(_boltz_phase2a_dir, exist_ok=True)

        # ══════════════════════════════════════════════════════════════════════
        # Step 2a-i : Protenix (cuda:1) — run ALL targets before touching Boltz
        # ══════════════════════════════════════════════════════════════════════
        print(f"\n── 2a-i: Protenix (cuda:1) ──")
        work_dir = Path("/kaggle/working")
        work_dir.mkdir(parents=True, exist_ok=True)

        queue_tids      = set(standard_queue.keys())
        queue_df        = (test_df_trunc[test_df_trunc["target_id"].isin(queue_tids)]
                           .reset_index(drop=True))
        input_json_path = str(work_dir / "protenix_queue_input.json")
        build_input_json(queue_df, input_json_path)

        from protenix.data.inference.infer_dataloader import InferenceDataset
        from runner.inference import (InferenceRunner,
                                      update_gpu_compatible_configs,
                                      update_inference_configs)

        configs = build_configs(input_json_path, str(work_dir / "outputs"), MODEL_NAME)
        configs.checkpoint_path = PROTENIX_CHECKPOINT
        print(f"✅ Protenix checkpoint: {configs.checkpoint_path}")
        configs = update_gpu_compatible_configs(configs)
        runner  = InferenceRunner(configs)
        dataset = InferenceDataset(configs)

        for i in tqdm(range(len(dataset)), desc="Protenix"):
            data, atom_array, error_message = dataset[i]
            target_id = data.get("sample_name", f"sample_{i}")
            if target_id not in _split:
                continue
            n_ptx, n_boltz, n_needed, full_seq = _split[target_id]
            if error_message:
                print(f"  {target_id}: data error — {error_message}")
                protenix_preds[target_id] = None
                del data, atom_array, error_message
                gc.collect(); torch.cuda.empty_cache(); gc.collect()
                continue
            try:
                new_cfg = update_inference_configs(configs, data["N_token"].item())
                # Oversample (2× Protenix quota + 1) for diversity, then trim
                # down to exactly n_ptx via pick_diverse.
                new_cfg.sample_diffusion.N_sample = max(8, n_ptx * 2 + 1)
                runner.update_model_configs(new_cfg)
                prediction = runner.predict(data)
                raw_coords = prediction["coordinate"]
                feat = data["input_feature_dict"]
                if "centre_atom_mask" in feat:
                    mask = (feat["centre_atom_mask"] == 1).to(raw_coords.device)
                elif "atom_to_tokatom_idx" in feat:
                    m11 = (feat["atom_to_tokatom_idx"] == 11).to(raw_coords.device)
                    m12 = (feat["atom_to_tokatom_idx"] == 12).to(raw_coords.device)
                    mask = (m11 if abs(m11.sum() - len(full_seq)) <
                                   abs(m12.sum() - len(full_seq)) else m12)
                else:
                    mask = torch.zeros(raw_coords.shape[1], dtype=torch.bool,
                                       device=raw_coords.device)
                coords = raw_coords[:, mask, :].detach().cpu().numpy()
                if coords.shape[1] != len(full_seq):
                    padded  = np.zeros((coords.shape[0], len(full_seq), 3),
                                       dtype=np.float32)
                    min_len = min(coords.shape[1], len(full_seq))
                    if min_len > 0:
                        padded[:, :min_len, :] = coords[:, :min_len, :]
                    coords = padded
                if coords.shape[1] > 1:
                    diffs = np.linalg.norm(coords[0, 1:] - coords[0, :-1], axis=-1)
                    if np.all(diffs < 1e-4):
                        print(f"  {target_id}: collapsed — zeroing")
                        coords = np.zeros_like(coords)
                # Greedy max-min diversity trim to Protenix quota.
                coords = pick_diverse(coords, min(n_ptx, coords.shape[0]))
                protenix_preds[target_id] = coords
                print(f"  {target_id}: {coords.shape[0]} Protenix preds "
                      f"(quota={n_ptx})")
            except Exception as exc:
                print(f"  {target_id}: Protenix FAILED — {exc}")
                import traceback; traceback.print_exc()
                protenix_preds[target_id] = None
            finally:
                # Guard each del individually — variable may not be bound if
                # the exception fired before its assignment.
                for _vn in ["prediction", "raw_coords", "mask"]:
                    try:    exec(f"del {_vn}")
                    except: pass
                del data, atom_array
                gc.collect(); torch.cuda.empty_cache(); gc.collect()

        # ══════════════════════════════════════════════════════════════════════
        # Step 2a-ii : VRAM sweep — release ALL Protenix tensors before
        #              Boltz-1 is allowed to allocate on cuda:1.
        # ══════════════════════════════════════════════════════════════════════
        print("\n── 2a-ii: VRAM sweep (Protenix complete) ──")
        for _vn in ["runner", "dataset", "configs"]:
            try:    exec(f"del {_vn}")
            except: pass
        gc.collect()
        torch.cuda.empty_cache()
        gc.collect()
        if torch.cuda.is_available() and torch.cuda.device_count() > 1:
            _free_mb, _tot_mb = torch.cuda.mem_get_info(1)
            print(f"  cuda:1 after Protenix sweep: "
                  f"{_free_mb / 1024**3:.2f} / {_tot_mb / 1024**3:.2f} GB free")

        # ══════════════════════════════════════════════════════════════════════
        # Step 2a-iii : Boltz-1 (cuda:1) — run sequentially, all targets
        # ══════════════════════════════════════════════════════════════════════
        print(f"\n── 2a-iii: Boltz-1 (cuda:1) ──")
        for _tid, (n_ptx, n_boltz, n_needed, full_seq) in _split.items():
            if n_boltz == 0:
                print(f"  {_tid}: n_boltz=0 — full quota assigned to Protenix")
                boltz_phase2a_preds[_tid] = []
                continue
            if len(full_seq) >= BOLTZ_MAX_LEN:
                print(f"  {_tid}: {len(full_seq)} nt ≥ {BOLTZ_MAX_LEN} "
                      f"— Boltz-1 OOM guard active, skipping")
                boltz_phase2a_preds[_tid] = [None] * n_boltz
                continue
            _out = os.path.join(_boltz_phase2a_dir, _tid)
            _seeds = generate_boltz_seeds_batch(
                target_id=_tid,
                sequence=full_seq,
                n_seeds=n_boltz,
                out_dir=_out,
                device=DEVICE_BOLTZ,
            )
            boltz_phase2a_preds[_tid] = _seeds

        # ══════════════════════════════════════════════════════════════════════
        # Step 2a-iv : Final VRAM sweep after Boltz-1 subprocess batch
        # ══════════════════════════════════════════════════════════════════════
        print("\n── 2a-iv: VRAM sweep (Boltz-1 complete) ──")
        gc.collect()
        torch.cuda.empty_cache()
        gc.collect()

        # ══════════════════════════════════════════════════════════════════════
        # Step 2a-v : Ensemble summary log
        # ══════════════════════════════════════════════════════════════════════
        print(f"\n── Phase 2a Ensemble Summary ──")
        for _tid in standard_queue:
            _ptx_arr   = protenix_preds.get(_tid)
            _boltz_lst = boltz_phase2a_preds.get(_tid) or []
            _n_p = (_ptx_arr.shape[0]
                    if _ptx_arr is not None and hasattr(_ptx_arr, "shape")
                    else 0)
            _n_b = sum(1 for _s in _boltz_lst if _s is not None)
            _, _, _nn, _ = _split[_tid]
            print(f"  {_tid}: {_n_p} Protenix preds, {_n_b} Boltz preds "
                  f"(total {_n_p + _n_b} / {_nn})")

    # ── Phase 2b: SHR Megascale ───────────────────────────────────────────────
    megascale_preds = {}
    if megascale_targets:
        print(f"\n{'='*60}")
        print(f"PHASE 2b: SHR Megascale for {len(megascale_targets)} targets")
        print(f"{'='*60}")
        for tid, (n_needed, full_seq) in megascale_targets.items():
            try:
                megascale_preds[tid] = handle_megascale_target(
                    tid, full_seq, segments_map, train_coords, combined_seqs
                )
            except Exception as exc:
                print(f"  {tid}: MEGASCALE FAILED — {exc}")
                import traceback; traceback.print_exc()
                megascale_preds[tid] = None

    # ── Phase 3: SHR Polish (blended contact maps) ────────────────────────────
 # ── Phase 3: SHR Physics Polish ──────────────────────────────────────────
    if SHR_ENABLED and (SHR_POLISH_TBM or SHR_POLISH_PTX):
        print(f"\n{'='*60}")
        print("PHASE 3: SHR Physics Polish (RNA-FM ⊕ RibonanzaNet-2 maps)")
        print(f"{'='*60}")
        
        if SHR_POLISH_TBM:
            from scipy.spatial import KDTree  # Ensure KDTree is available
            for tid, preds_list in template_preds.items():
                seq = seq_by_id.get(tid, "")
                if not seq or not preds_list:
                    continue
                cmap = get_or_build_contact_map(seq, tid, train_coords, combined_seqs, segments_map)
                
                for j in range(len(preds_list)):
                    if preds_list[j] is not None and not np.allclose(preds_list[j], 0.0):
                        _p = preds_list[j].copy()

                        # ── v15: Slot-0 Crystal Guard ────────────────────────────────────
                        # j == 0 is the pristine crystal template (slot 0 purity).
                        # Subjecting a fully resolved crystal structure to 8,000 JAX
                        # steps creates artificial steric explosions (-1500 A clips).
                        # Preserve it exactly as-is; only j >= 1 gets physics polish.
                        if j == 0:
                            preds_list[0] = _p   # retain raw crystal coords
                            continue

                        # ── 1. Zero-mask jitter (Catch uninitialized atoms) ──
                        _zm = np.all(_p == 0.0, axis=1)
                        if _zm.any():
                            _p[_zm] = np.random.default_rng(j).normal(
                                0.0, 2.0, (_zm.sum(), 3)
                            ).astype(np.float32)
                            
                                                # ── 2. KDTree Armor — ABSOLUTE jitter on non-sequential clashes ─────
                        # CRITICAL: use absolute placement (not +=).  Atoms that are truly
                        # coincident need displacement to a NEW position; adding a delta to
                        # (0,0,0) + (0,0,0) = (0,0,0) achieves nothing.
                        # Absolute placement guarantees min inter-atom distance > ARMOR_R
                        # before the JAX scan starts, eliminating divide-by-zero gradients.
                        try:
                            from scipy.spatial import KDTree as _KT3
                            _ARMOR_R = 1.5   # Angstroms — matches megascale Stage-2 threshold
                            _close3  = _KT3(_p).query_pairs(_ARMOR_R)
                            _cset3   = set()
                            for _pa3, _pb3 in _close3:
                                if abs(_pa3 - _pb3) > 1:   # skip sequential backbone pairs
                                    _cset3.add(_pa3)
                                    _cset3.add(_pb3)
                            if _cset3:
                                _cidx3   = np.array(sorted(_cset3), dtype=int)
                                _rng3    = np.random.default_rng(42 + j)
                                # Scatter clashing atoms radially from their centroid
                                # to guaranteed-safe positions 3-6 A apart
                                _center3 = _p[_cidx3].mean(axis=0)
                                _dirs3   = _rng3.normal(0.0, 1.0, (len(_cidx3), 3))
                                _dirs3  /= np.linalg.norm(_dirs3, axis=1, keepdims=True) + 1e-8
                                _radii3  = _rng3.uniform(3.0, 6.0, (len(_cidx3), 1))
                                _p[_cidx3] = _center3 + _dirs3 * _radii3   # ABSOLUTE
                                print(f"  [KDTree Armor] {tid} pred {j}: "
                                      f"displaced {len(_cidx3)} atoms "
                                      f"({len(_close3)} pairs < {_ARMOR_R} A)")
                        except Exception as _ka_exc:
                            # Non-fatal: log and continue — polish may still succeed
                            print(f"  [KDTree Armor] WARNING: {_ka_exc} — skipping armor")
                            
                        # ── 3. Run Polish ──
                        try:
                            preds_list[j] = shr_polish(_p, contact_map=cmap, n_steps=SHR_POLISH_STEPS)
                        except Exception as e:
                            print(f"  {tid}: Polish failed ({e}), keeping raw template.")
                            preds_list[j] = _p
                            
                print(f"  {tid}: polished {len(preds_list)} TBM predictions")

        if SHR_POLISH_PTX:
            # ── Polish Protenix Phase-2a seeds ───────────────────────────────────────
            for tid, ptx in protenix_preds.items():
                if ptx is None or ptx.ndim != 3:
                    continue
                seq = seq_by_id.get(tid, "")
                if not seq:
                    continue
                cmap = get_or_build_contact_map(seq, tid, train_coords, combined_seqs, segments_map)
                for j in range(ptx.shape[0]):
                    if not np.allclose(ptx[j], 0.0):
                        # ── v18 Fix: Protenix Crystal Guard ──
                        if j == 0: 
                            print(f"  {tid}: Skipping polish for Model 1 (Guard active)")
                            continue 
                        ptx[j] = shr_polish(ptx[j], contact_map=cmap, n_steps=SHR_POLISH_STEPS)
                print(f"  {tid}: polished {ptx.shape[0]} Protenix predictions")
                
            # ── Polish Boltz-1 Phase-2a seeds ───────────────────────────────────────
            for tid, boltz_list in boltz_phase2a_preds.items():
                if not boltz_list:
                    continue
                seq = seq_by_id.get(tid, "")
                if not seq:
                    continue
                cmap = get_or_build_contact_map(seq, tid, train_coords, combined_seqs, segments_map)
                n_polished = 0
                for j, _bcoords in enumerate(boltz_list):
                    if _bcoords is not None and not np.allclose(_bcoords, 0.0):
                        # ── v18 Fix: Boltz-1 Crystal Guard ──
                        if j == 0: continue 
                        boltz_list[j] = shr_polish(
                            _bcoords, contact_map=cmap, n_steps=SHR_POLISH_STEPS
                        )
                        n_polished += 1
                print(f"  {tid}: polished {n_polished} Boltz-1 Phase-2a predictions")

    # ── Phase 4: Assemble TBM + Protenix/Boltz-1 Ensemble + Megascale + de-novo ─
    print(f"\n{'='*60}")
    print("PHASE 4: Assemble TBM + Protenix/Boltz-1 Ensemble + Megascale + de-novo")
    print(f"{'='*60}")

    _boltz_out_dir = "/kaggle/working/boltz_outputs"
    os.makedirs(_boltz_out_dir, exist_ok=True)
    all_rows = []

    for _, row in test_df.iterrows():
        tid = row["target_id"]
        seq = row["sequence"]

        # Start with TBM seeds
        combined = list(template_preds.get(tid, []))

        # ── v16: Megascale SHR seeds (dedup-aware) ────────────────────────
        mega = megascale_preds.get(tid)
        if mega is not None and mega.ndim == 3:
            for j in range(mega.shape[0]):
                # v16: If TBM already provided the crystal template as Model 1,
                # Megascale Seed 0 is an identical copy (Crystal Guard preserved
                # it verbatim).  Skip it to avoid wasting a slot on a duplicate
                # and dropping the 5th compacted diversity model.
                if j == 0 and len(combined) > 0:
                    continue
                if len(combined) >= N_SAMPLE:
                    break
                combined.append(mega[j])

        # ── Protenix Phase-2a seeds ───────────────────────────────────────────────
        ptx = protenix_preds.get(tid)
        n_ptx_added = 0
        if ptx is not None and ptx.ndim == 3:
            for j in range(ptx.shape[0]):
                if len(combined) >= N_SAMPLE:
                    break
                combined.append(ptx[j])
                n_ptx_added += 1

        # ── Boltz-1 Phase-2a seeds (pre-generated in 2a-iii, SHR-polished in Phase 3)
        _b2a_list = boltz_phase2a_preds.get(tid) or []
        n_boltz_added = 0
        for _bc in _b2a_list:
            if len(combined) >= N_SAMPLE:
                break
            if _bc is not None and len(_bc) == len(seq):
                combined.append(_bc)
                n_boltz_added += 1
            elif _bc is not None:
                print(f"  {tid}: Boltz Phase-2a coord length mismatch "
                      f"({len(_bc)} vs {len(seq)}) — discarded")
        if n_ptx_added or n_boltz_added:
            print(f"  {tid}: {n_ptx_added} Protenix preds, "
                  f"{n_boltz_added} Boltz preds (Phase 2a ensemble)")

        # ── Boltz-1 gap-filler (GPU 1, only if N_SAMPLE slots are still unfilled) ──
        if len(combined) < N_SAMPLE and len(seq) < BOLTZ_MAX_LEN:
            cmap = get_or_build_contact_map(
                seq, tid, train_coords, combined_seqs, segments_map
            )
            _boltz_coords = generate_boltz_seed(
                tid, seq,
                out_dir=os.path.join(_boltz_out_dir, tid),
                device=DEVICE_BOLTZ
            )
            if _boltz_coords is not None and len(_boltz_coords) == len(seq):
                # SHR-polish the Boltz seed with blended contact map
                _polished_boltz = shr_polish(
                    _boltz_coords, contact_map=cmap, n_steps=SHR_POLISH_STEPS
                )
                combined.append(_polished_boltz)
                print(f"  {tid}: Boltz-1 seed polished & added ✅")
            elif _boltz_coords is not None:
                print(f"  {tid}: Boltz-1 coord length mismatch "
                      f"({len(_boltz_coords)} vs {len(seq)}) — discarded")

        # ── De-novo fallback ──────────────────────────────────────────────────
        n_denovo = 0
        while len(combined) < N_SAMPLE:
            seed_val = row.name * 1000000 + len(combined) * 1000
            dn = generate_rna_structure(seq, seed=seed_val)
            dn = adaptive_rna_constraints(dn, tid, segments_map, confidence=0.2)
            if SHR_ENABLED and len(seq) < 2000:
                cmap = get_or_build_contact_map(
                    seq, tid, train_coords, combined_seqs, segments_map
                )
                dn = shr_polish(dn, contact_map=cmap, n_steps=SHR_POLISH_STEPS)
            combined.append(dn)
            n_denovo += 1

        if n_denovo:
            print(f"  {tid}: {n_denovo} slot(s) filled with de-novo + SHR")

        # ── v19 MEGASCALE ARMOR: Ensemble Alignment + Explosion Guard ─────
        # Model 1 (Crystal Guard) is NEVER modified.
        # Models 2-5 are Kabsch-aligned onto Model 1, then explosion-checked.
        # Cross-model sanitizer ensures grader consistency across all 5 slots.
        models_list = [c.copy() for c in combined[:N_SAMPLE]]
        print(f"  {tid}: Aligning {len(models_list)} models (Crystal Guard = Slot 0)")
        models_list = align_ensemble_to_slot0(models_list)
        models_list = sanitize_coordinates(models_list)
        all_rows.extend(coords_to_rows_v18(tid, seq, models_list))
# ── Save submission ───────────────────────────────────────────────────────
    # ── Save submission ───────────────────────────────────────────────────────
    sub = pd.DataFrame(all_rows)
    cols = ["ID", "resname", "resid"] + [
        f"{c}_{i}" for i in range(1, N_SAMPLE + 1) for c in ["x", "y", "z"]
    ]
    coord_cols = [c for c in cols if c.startswith(("x_", "y_", "z_"))]

    # ── Step 1 (v19): Trust the sanitizer — NO blind clipping ──────────────
    # v18 clipped ALL coordinates to [-1500, 1500] Å, which created a
    # "pancake" artifact on megascale targets where the stitcher launched
    # coordinates thousands of Å away — the clip squashed them to ±1500
    # instead of sentineling them.  The v19 sanitize_coordinates() function
    # already caught these explosions and replaced them with SENTINEL.
    # We only flag any remaining stragglers that somehow escaped.
    _remaining_explosions = (np.abs(sub[coord_cols]) > 1499.0) & (sub[coord_cols] > SENTINEL + 1.0)
    _n_remaining = int(_remaining_explosions.sum().sum())
    if _n_remaining > 0:
        print(f"  ⚠️  {_n_remaining} coordinates still > 1499 Å after sanitizer — sentineling")
        # Sentinel the entire residue for each affected prediction slot
        for _pred_i in range(1, N_SAMPLE + 1):
            _xc, _yc, _zc = f"x_{_pred_i}", f"y_{_pred_i}", f"z_{_pred_i}"
            _bad_row = ((np.abs(sub[_xc]) > 1499.0) & (sub[_xc] > SENTINEL + 1.0)) | \
                       ((np.abs(sub[_yc]) > 1499.0) & (sub[_yc] > SENTINEL + 1.0)) | \
                       ((np.abs(sub[_zc]) > 1499.0) & (sub[_zc] > SENTINEL + 1.0))
            if _bad_row.any():
                sub.loc[_bad_row, [_xc, _yc, _zc]] = -2000000.0
    else:
        print("  ✅ No coordinates > 1499 Å — sanitizer caught all explosions")

    # ── Step 2: Convert -999.999 dummy atoms to -2e6 for US-align grader ─────
    # np.isclose handles any fp noise introduced by Step 1 for values that were
    # already at -999.999 (they remain at -999.999 after clipping, so isclose
    # still fires correctly).
    print("Converting -999.999 dummy atoms to -2,000,000.0 for US-align grader ...")
    sub[coord_cols] = sub[coord_cols].where(
        ~np.isclose(sub[coord_cols], -999.999),
        other=-2000000.0
    )

    # ── Step 3 (FIX 2.1): Convert all-zero coordinate triplets to -2e6 ─────
    # Unresolved stitcher padding slots initialise to (0, 0, 0) via np.zeros.
    # The grader threshold is x > -1e17, so a (0,0,0) triple is treated as a
    # real residue and inflates the TM-score denominator.  Converting them here
    # prevents phantom atoms from degrading the score on megascale targets.
    # Note: a real C1' atom at the crystallographic origin is vanishingly rare
    # and would need all three coords to be exactly 0.000 post-clipping.
    print("Converting all-zero coordinate triplets to -2,000,000.0 (unresolved padding) ...")
    for _pred_i in range(1, N_SAMPLE + 1):
        _xc, _yc, _zc = f"x_{_pred_i}", f"y_{_pred_i}", f"z_{_pred_i}"
        _zero_mask = (sub[_xc] == 0.0) & (sub[_yc] == 0.0) & (sub[_zc] == 0.0)
        if _zero_mask.any():
            sub.loc[_zero_mask, [_xc, _yc, _zc]] = -2000000.0
            print(f"  Prediction {_pred_i}: {_zero_mask.sum()} all-zero residues flagged")

    # ── Step 4 (v14): NaN / Inf Sanitization — grader crash prevention ─────────
    # With kdl up to 25.0, JAX may emit NaN/Inf on catastrophic compaction.
    # The US-align grader throws "Scoring Error" on the ENTIRE submission if it
    # encounters even one non-finite float.  Replace all non-finite values with
    # the grader dummy sentinel -2,000,000.0.  The grader ignores atoms where
    # x < -1e17 and scores all remaining residues normally.
    print("Sanitizing NaN / Inf in coordinate columns ...")
    _n_nan_v14 = int(sub[coord_cols].isna().sum().sum())
    _n_inf_v14 = int(np.isinf(sub[coord_cols].values).sum())
    if _n_nan_v14 or _n_inf_v14:
        print(f"  ⚠️  {_n_nan_v14} NaN + {_n_inf_v14} Inf detected "
              f"-> replaced with -2,000,000.0")
    sub[coord_cols] = sub[coord_cols].replace([np.inf, -np.inf], -2000000.0)
    sub[coord_cols] = sub[coord_cols].fillna(-2000000.0)
    _bad_v14 = int((np.isinf(sub[coord_cols].values) |
                    np.isnan(sub[coord_cols].values)).sum())
    assert _bad_v14 == 0, f"Non-finite values remain after sanitization: {_bad_v14}"
    print("  ✅ All coordinate columns are finite.")

    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    sub[cols].to_csv(output_csv, index=False)
    print(f"\n✅ Submission saved: {output_csv}  ({len(sub):,} rows)")
    print("✅ V19-MEGASCALE-ARMOR SUBMISSION SAVED")


In [24]:
if __name__ == "__main__":
    main()


PHASE 0: Pre-loading contact models on GPU 0
Loading RNA-FM to cuda:0...


/kaggle/input/notebooks/venkatapadavala/rna-fm/RNA-FM/fm/pretrained.py:50: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  model_data = torch.load(model_location, map_location='cpu')
/kaggle/input/notebooks/venkatapadavala/rna-fm/RNA-FM/fm/pretrained.py:53: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  regression_data = torch.load(regression_location, map_location='cpu')


✅ RNA-FM loaded successfully to [cuda:0]
constructing 48 ConvTransformerEncoderLayers
✅ RibonanzaNet-2 loaded (387 MB) → cuda:0

Test targets: 28

Loading training data for TBM ...
Template pool: 5744 seqs, 5744 structures

PHASE 1: Template-Based Modeling
  MIN_SIMILARITY = 0.0  |  MIN_PCT_IDENTITY = adaptive
  8ZNQ (30 nt, min_pct=45.0): all 5 from TBM ✓ (best_sim=1.000)
  9IWF (69 nt, min_pct=45.0): all 5 from TBM ✓ (best_sim=1.000)
  9JGM (210 nt, min_pct=55.0): 3 TBM (best_sim=1.000) → queuing 2 Protenix samples
  9MME (4640 nt, min_pct=55.0): 1 TBM (best_sim=1.000) → queuing 4 Protenix samples
  9J09 (214 nt, min_pct=55.0): 1 TBM (best_sim=1.000) → queuing 4 Protenix samples
  9E9Q (101 nt, min_pct=55.0): 4 TBM (best_sim=1.000) → queuing 2 Protenix samples
  9CFN (59 nt, min_pct=45.0): all 5 from TBM ✓ (best_sim=1.000)
  9OBM (73 nt, min_pct=45.0): all 5 from TBM ✓ (best_sim=1.000)
  9G4P (68 nt, min_pct=45.0): all 5 from TBM ✓ (best_sim=1.000)
  9G4Q (104 nt, min_pct=55.0): 2 TB

2026-03-05 04:58:34,802 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/runner/inference.py:557] INFO runner.inference: Enforcing FP32 and torch kernels for compatibility with detected GPU (Compute Capability 7.x).
2026-03-05 04:58:34,803 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/runner/inference.py:246] INFO runner.inference: Distributed environment: world size: 1, global rank: 0, local rank: 0
2026-03-05 04:58:34,804 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/runner/inference.py:98] INFO root: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
2026-03-05 04:58:34,805 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/runner/inference.py:127] INFO root: Finished environment initialization.


  9ZCC (1460 nt, min_pct=55.0): 1 TBM (best_sim=1.000) → queuing 4 Protenix samples

Phase 1 done in 594.7s
  Fully covered by TBM : 11
  Routed to Protenix   : 17
  9MME: MEGASCALE (4640 nt) → SHR
  9ZCC: MEGASCALE (1460 nt) → SHR

PHASE 2a: Protenix + Boltz-1 Ensemble for 15 standard targets
  9JGM (210 nt): quota=2 → 1 Protenix + 1 Boltz-1
  9J09 (214 nt): quota=4 → 2 Protenix + 2 Boltz-1
  9E9Q (101 nt): quota=2 → 1 Protenix + 1 Boltz-1
  9G4Q (104 nt): quota=3 → 2 Protenix + 1 Boltz-1
  9G4R (47 nt): quota=4 → 2 Protenix + 2 Boltz-1
  9RVP (34 nt): quota=2 → 1 Protenix + 1 Boltz-1
  9JFS (246 nt): quota=4 → 2 Protenix + 2 Boltz-1
  9LEC (378 nt): quota=3 → 2 Protenix + 1 Boltz-1
  9LEL (476 nt): quota=3 → 2 Protenix + 1 Boltz-1
  9JFO (195 nt): quota=4 → 2 Protenix + 2 Boltz-1
  9WHV (80 nt): quota=4 → 2 Protenix + 2 Boltz-1
  9E74 (255 nt): quota=3 → 2 Protenix + 1 Boltz-1
  9E75 (165 nt): quota=2 → 1 Protenix + 1 Boltz-1
  9KGG (267 nt): quota=2 → 1 Protenix + 1 Boltz-1
  9EBP (

...

  9EBP: 2 Protenix preds (quota=2)


Protenix: 100%|██████████| 15/15 [43:27<00:00, 173.83s/it]



── 2a-ii: VRAM sweep (Protenix complete) ──
  cuda:1 after Protenix sweep: 14.46 / 14.56 GB free

── 2a-iii: Boltz-1 (cuda:1) ──
  [Boltz] 9JGM seed 1/1 on cuda:1 (210 nt) ...
  [Boltz] ✅ seed 0: 210 C1' atoms from 9JGM_b0_model_0.cif
  [Boltz batch] 9JGM: 1/1 seeds collected
  [Boltz] 9J09 seed 1/2 on cuda:1 (214 nt) ...
  [Boltz] ✅ seed 0: 214 C1' atoms from 9J09_b0_model_0.cif
  [Boltz] 9J09 seed 2/2 on cuda:1 (214 nt) ...
  [Boltz] ✅ seed 1: 214 C1' atoms from 9J09_b1_model_0.cif
  [Boltz batch] 9J09: 2/2 seeds collected
  [Boltz] 9E9Q seed 1/1 on cuda:1 (101 nt) ...
  [Boltz] ✅ seed 0: 101 C1' atoms from 9E9Q_b0_model_0.cif
  [Boltz batch] 9E9Q: 1/1 seeds collected
  [Boltz] 9G4Q seed 1/1 on cuda:1 (104 nt) ...
  [Boltz] ✅ seed 0: 104 C1' atoms from 9G4Q_b0_model_0.cif
  [Boltz batch] 9G4Q: 1/1 seeds collected
  [Boltz] 9G4R seed 1/2 on cuda:1 (47 nt) ...
  [Boltz] ✅ seed 0: 47 C1' atoms from 9G4R_b0_model_0.cif
  [Boltz] 9G4R seed 2/2 on cuda:1 (47 nt) ...
  [Boltz] ✅ seed 1: 47

INFO:2026-03-05 06:49:33,503:jax._src.xla_bridge:822: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
2026-03-05 06:49:33,503 [/usr/local/lib/python3.12/dist-packages/jax/_src/xla_bridge.py:822] INFO jax._src.xla_bridge: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory


  8ZNQ: polished 5 TBM predictions
  [HWS] Contact map: 69x69, 1953 contacts (threshold=0.85)
  9IWF: RNA-FM HWS contact map (69×69) ✅
  [RibonanzaNet-2] 69×69 contact map, 614 pairs > 0.5
  9IWF: blended (RNA-FM 50% + RibonanzaNet-2 HWS 50%)
  9IWF: polished 5 TBM predictions
  [HWS] Contact map: 210x210, 20706 contacts (threshold=0.85)
  9JGM: RNA-FM HWS contact map (210×210) ✅
  [RibonanzaNet-2] 210×210 contact map, 11678 pairs > 0.5
  9JGM: blended (RNA-FM 50% + RibonanzaNet-2 HWS 50%)
  [KDTree Armor] 9JGM pred 2: displaced 11 atoms (21 pairs < 1.5 A)
  9JGM: polished 3 TBM predictions
  9MME: polished 1 TBM predictions
  [HWS] Contact map: 214x214, 21528 contacts (threshold=0.85)
  9J09: RNA-FM HWS contact map (214×214) ✅
  [RibonanzaNet-2] 214×214 contact map, 11441 pairs > 0.5
  9J09: blended (RNA-FM 50% + RibonanzaNet-2 HWS 50%)
  9J09: polished 1 TBM predictions
  [HWS] Contact map: 101x101, 4465 contacts (threshold=0.85)
  9E9Q: RNA-FM HWS contact map (101×101) ✅
  [Ribonanz